# Introduction

## The research question

**When my season-total projection model and Sleeper's preseason projection both rank a player on the
same side of his ADP, does he finish on that side?**

ADP — average draft position — is the market's price on a player. A projection that ranks him
*above* his ADP is calling him underpriced; *below*, overpriced. This notebook asks whether the two
projections **agreeing** with each other in that disagreement with the market identifies players who
actually finish where they said.

There are two very different claims hiding inside that question, and separating them is the whole
point of this study:

1. **Does the agreement cell beat chance?** Measured against an empirical null built by shuffling
   outcomes within season and position (Stage 8a).
2. **Does *my model* add anything to Sleeper alone?** Sleeper's projection is free and public. If
   agreement works only because Sleeper works, my model contributes nothing (Stage 8b).

A study that answers (1) and reports it as if it answered (2) would be wrong in the most useful
direction — which is exactly why they are computed and reported separately, and why the verdict
below rests on (2).

## Status — read before quoting anything

This is **descriptive, post-hoc research**, run on 2026-08-02 at Joseph's request. It is **not
pre-registered** and **not live-validated**. The threshold grid `{0, >5, >7.5, >10}` was fixed by the
request before any result was seen, and every cell at every threshold is reported — including the
ones that fail. No threshold was selected after the fact, and none may be.

## Inputs

Read-only, from their existing production locations. Nothing is copied into this research folder and
no model is re-run, re-fit, or re-tuned.

| Input | Role |
|---|---|
| `fantasy/projections/results/{,wr_,te_,qb_}walkforward_predictions.csv` | The shipped models' walk-forward predictions and observed season totals, 2021–2025 |
| `fantasy/seasonal_projections/season_dataset_2014_2025.csv` | ADP, team, identity, and the dataset's own Sleeper column |
| `fantasy/projections/build_{rb,wr,te,qb}_projection.py` | The generating code, audited statically in Stage 2 |
| `fantasy/seasonal_projections/h11_freshness_signal.py` + its staged Underdog dumps | The dated-market freshness sensitivity (Stage 10) |

"My model" means the raw walk-forward `pred` column and nothing else — not the retired seasonal
model, not the old blended board, not the 2026 fitted values, and not
`analyst_projection_adjustments_2026.csv`.

## Populations, universes, and why both are carried

**Populations.** `all_adp` is every walk-forward row carrying an ADP. `drafted_top180` restricts to
`adp_overall_rank <= 180` — the repo's own draftable-universe convention
(`phase0_benchmark.POOL_SIZE`). Beyond roughly pick 180 nobody in a 12-team league is drafting, so
"ADP" out there is a price no one paid. **This split turns out to be the single most consequential
choice in the study.**

**Rank universes.** `A` is the production-board analogue: rank over the whole ADP-bearing model
population, with a missing Sleeper projection leaving a missing Sleeper rank. `B` is the required
sensitivity: restrict to rows complete on all four quantities first, then re-rank inside that
identical set. A conclusion that moves between A and B must be reported as population-sensitive.

## Exclusions

- **QB rookies.** The QB rookie arm was fitted and then held back from the shipped surface, so
  scoring it would credit predictions the product does not make. RB/WR/TE rookies are kept — those
  arms shipped.
- **Season 2020.** Its stored Sleeper artifact is provenance-contaminated (near-actuals), quarantined
  at source in the seasonal campaign.
- **Nothing else.** In particular, injuries and games missed are *not* filtered. Both systems
  forecast season totals, so availability is part of the quantity being predicted.

## Pipeline map

| Stage | Cells | What it does |
|---|---|---|
| 1 | Config, hashes | Pin every parameter; SHA-256 every input; compare against the archived run |
| 2 | Generator audit | Read the generating source and prove season *Y* trained only on seasons `< Y` |
| 3 | Load, join, filter | Per-file integrity, exact `(season, player_id)` join, population filter |
| 4 | Ranks | Why stored `adp_pos_rank` is unusable; build both universes x both populations |
| 5 | Signals | Gaps, agreement, strict thresholds, consensus score, hit/miss/tie |
| 6 | Main results | Pooled 2024–2025 both ways; then test the undrafted-tail artifact directly |
| 7 | Stability | Pooled panels, per season, per position, veteran vs rookie |
| 8 | Inference | Permutation null (beats chance?) then bootstrap lifts (adds value beyond Sleeper?) |
| 9 | Logistic | Does agreement matter beyond gap magnitude? |
| 10 | Freshness | The dated Underdog market, reconstructed offline |
| 11 | Audit | Individual drafted-board calls, hits and misses |
| 12 | Export & checks | Write artifacts, reconstruct summaries from row data, re-hash, self-audit structure |

## Outputs

Written to `artifacts/`: `threshold_summary.csv`, `player_season_results.csv`,
`incremental_comparisons.csv`, `manifest.json`. These are **machine-readable outputs of this
notebook**, not a second source of truth — every definition, result, caveat and conclusion lives in
the notebook itself. The original 2026-08-02 run is preserved byte-for-byte under
`archive/original_2026-08-02/`.

## How to read this notebook

Every code cell sits between a `### Explain` cell that says what it is about to do and why, and a
`### Interpretation` cell that reads the output it produced. The Interpretation cells quote the
numbers from the recorded run — if a number in prose disagrees with the output above it, the output
is right and the prose is stale.

### Explain — Stage 1a: resolve the repository root and pin study configuration

This cell fixes everything the study is allowed to vary, before any data is read, so that no
parameter can be chosen after seeing a result.

**Inputs.** None from disk. The repository root is *discovered* by walking up from the notebook's
own location until a directory containing both `CLAUDE.md` and `fantasy/projections` is found, so
every downstream path is relative to the repo and the notebook survives the repository being
renamed or relocated.

**What it defines.**

- `TEST_SEASONS` — 2021–2025. This is the range where the walk-forward predictions exist *and*
  Sleeper projections are available. 2020 is absent by design: its stored Sleeper artifact is
  provenance-contaminated (near-actuals), quarantined at source in the seasonal campaign.
- `THRESHOLDS` — `[0, 5, 7.5, 10]`, applied with a strict `>`. Ranks are integers, so `>7.5`
  means **at least 8 spots**. This grid was fixed by the request; nothing is added or dropped later.
- `PANELS` — the five individual seasons plus the three pooled panels. All are computed and all
  are reported.
- `POPULATIONS` — `all_adp` (every walk-forward row carrying an ADP) and `drafted_top180`
  (`adp_overall_rank <= 180`, the repo's own draftable-universe convention from
  `fantasy/seasonal_projections/phase0_benchmark.py`). Carrying both is not optional: the split
  between them turns out to be the single most important result in this study.
- `SEED`, `N_PERM`, `N_BOOT` — the resampling budget and the one seed that drives every
  permutation and bootstrap draw, so the notebook is bit-reproducible.

**Output.** A printed configuration record — resolved paths, existence checks, and every pinned
parameter — so the Interpretation below is grounded in observed state rather than intent.

**Assumption being asserted:** the repo root discovery must succeed and every declared input path
must already exist. Both are hard assertions; a missing input stops the notebook here rather than
producing a partial study.

In [1]:
import hashlib, json, sys, platform, textwrap
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def _find_repo_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "CLAUDE.md").exists() and (cand / "fantasy" / "projections").is_dir():
            return cand
    raise RuntimeError(f"repository root not found above {start}")


NB_DIR = Path.cwd().resolve()
REPO = _find_repo_root(NB_DIR)
PROJECT = REPO / "fantasy" / "projections" / "research" / "adp_consensus_agreement_2026-08-02"
ARTIFACTS = PROJECT / "artifacts"
ARCHIVE = PROJECT / "archive" / "original_2026-08-02"
NOTEBOOK_PATH = PROJECT / "adp_consensus_pipeline.ipynb"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RESULTS = REPO / "fantasy" / "projections" / "results"
SEAS_DIR = REPO / "fantasy" / "seasonal_projections"
SEAS_CSV = SEAS_DIR / "season_dataset_2014_2025.csv"
WF_FILES = {
    "RB": RESULTS / "walkforward_predictions.csv",
    "WR": RESULTS / "wr_walkforward_predictions.csv",
    "TE": RESULTS / "te_walkforward_predictions.csv",
    "QB": RESULTS / "qb_walkforward_predictions.csv",
}
BUILDERS = {p: REPO / "fantasy" / "projections" / f"build_{p.lower()}_projection.py"
            for p in ("RB", "WR", "TE", "QB")}

TEST_SEASONS = [2021, 2022, 2023, 2024, 2025]
THRESHOLDS = [0.0, 5.0, 7.5, 10.0]
PANELS = {
    "season_2021": [2021], "season_2022": [2022], "season_2023": [2023],
    "season_2024": [2024], "season_2025": [2025],
    "pooled_2024_2025": [2024, 2025],
    "pooled_2023_2025": [2023, 2024, 2025],
    "pooled_2021_2025": TEST_SEASONS,
}
POOLED_PANELS = ["pooled_2024_2025", "pooled_2023_2025", "pooled_2021_2025"]
POPULATIONS = {"all_adp": None, "drafted_top180": 180}
DRAFTABLE_POOL_SIZE = 180
SEED, N_PERM, N_BOOT = 20260802, 10_000, 10_000
RUN_TS = datetime.now(timezone.utc).isoformat(timespec="seconds")

_inputs = {**WF_FILES, "season_dataset": SEAS_CSV, **{f"builder_{k}": v for k, v in BUILDERS.items()}}
for label, p in _inputs.items():
    assert p.exists(), f"declared input missing: {label} -> {p}"

print("=" * 88)
print("STUDY CONFIGURATION — ADP-consensus agreement (descriptive, post-hoc)")
print("=" * 88)
print(f"run timestamp (UTC)   : {RUN_TS}")
print(f"python                : {sys.version.split()[0]}  |  numpy {np.__version__}  pandas {pd.__version__}")
print(f"platform              : {platform.platform()}")
print(f"repository root       : {REPO}")
print(f"project folder        : {PROJECT.relative_to(REPO)}")
print(f"active notebook       : {NOTEBOOK_PATH.relative_to(REPO)}")
print(f"artifacts out         : {ARTIFACTS.relative_to(REPO)}")
print(f"archive (original run): {ARCHIVE.relative_to(REPO)}")
print("-" * 88)
print(f"test seasons          : {TEST_SEASONS}   (2020 excluded: Sleeper artifact provenance-contaminated)")
print(f"thresholds (strict >) : {THRESHOLDS}   -> '>7.5' means at least 8 integer rank spots")
print(f"panels                : {len(PANELS)} ({', '.join(PANELS)})")
print(f"rank universes        : A = board analogue | B = common universe")
print(f"populations           : all_adp (no cap) | drafted_top180 (adp_overall_rank <= {DRAFTABLE_POOL_SIZE})")
print(f"seed / permutations / bootstraps : {SEED} / {N_PERM:,} / {N_BOOT:,}")
print("-" * 88)
print("declared inputs (all exist):")
for label, p in _inputs.items():
    print(f"  {label:18s} {str(p.relative_to(REPO)):58s} {p.stat().st_size:>10,} B")
print("=" * 88)

STUDY CONFIGURATION — ADP-consensus agreement (descriptive, post-hoc)
run timestamp (UTC)   : 2026-08-03T01:08:57+00:00
python                : 3.11.9  |  numpy 2.4.6  pandas 3.0.3
platform              : Windows-10-10.0.26200-SP0
repository root       : C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics
project folder        : fantasy\projections\research\adp_consensus_agreement_2026-08-02
active notebook       : fantasy\projections\research\adp_consensus_agreement_2026-08-02\adp_consensus_pipeline.ipynb
artifacts out         : fantasy\projections\research\adp_consensus_agreement_2026-08-02\artifacts
archive (original run): fantasy\projections\research\adp_consensus_agreement_2026-08-02\archive\original_2026-08-02
----------------------------------------------------------------------------------------
test seasons          : [2021, 2022, 2023, 2024, 2025]   (2020 excluded: Sleeper artifact provenance-contaminated)
thresholds (strict >) : [0.0, 5.0, 7.5, 10.0]   -> '>7.5' me

### Interpretation — configuration resolved, nothing left implicit

The repo root resolved to `JoSchoAnalytics` and all nine declared inputs exist, so the notebook is
correctly anchored despite the repository having been renamed from `BettingEdgeContinued` since the
original run. Every path printed above is derived from that root, so nothing here depends on an
absolute location.

The environment is Python 3.11.9 / numpy 2.4.6 / pandas 3.0.3. Worth noting because the seeded
resampling in Stages 8a–8b depends on numpy's `default_rng` stream: the p-values and CIs are
reproducible on this numpy, and a major-version change could shift the last digits without changing
any conclusion.

The parameter block is the substantive content. Eight panels x four thresholds x two universes x two
populations is 128 headline cells before any split, and **all of them are computed and exported**.
That matters for a post-hoc study: there is no scope to quietly report only the favourable slice,
because the full grid ships in `threshold_summary.csv` and anyone can check it.

Two choices already constrain the verdict. First, `TEST_SEASONS` starts at 2021, so 2020 never
enters — its Sleeper artifact is contaminated, and including it would flatter every Sleeper-related
number. Second, `POPULATIONS` carries `drafted_top180` alongside `all_adp`. Stage 6c is where that
second choice earns its place; for now it is simply pinned so it cannot be dropped later.

Next: hash these inputs so every number below is tied to specific bytes.

### Explain — Stage 1b: SHA-256 every input and record the provenance baseline

Reproducibility here means more than "the code runs". It means the numbers below are pinned to a
specific set of bytes. This cell hashes every source file the study reads and stores those digests
in `INPUT_HASHES`.

**Inputs.** The four walk-forward prediction CSVs (one per position), the season dataset that
supplies ADP and identity fields, and the four builder scripts that generated the walk-forward
files.

**Output.** A table of relative path, SHA-256, and byte size, plus a comparison against the digests
recorded by the original 2026-08-02 run (`archive/original_2026-08-02/manifest.json`). That
comparison is the check that matters: the repository was renamed from `BettingEdgeContinued` to
`JoSchoAnalytics` between the two runs, so this confirms the *rename moved files without altering
their contents* and that every recomputed number below is directly comparable to the archived run.

**Why hash the builders too.** The CSVs are outputs. The builder scripts are the logic that
produced them, and the next cell audits that logic directly. Hashing both means a future reader can
tell whether a discrepancy came from changed data or changed generating code.

`INPUT_HASHES` is re-verified at the very end of the notebook; anything that mutates an input
mid-run is caught there.

In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


INPUT_HASHES = {}
for label, p in {**{f"walkforward_{k}": v for k, v in WF_FILES.items()},
                 "season_dataset": SEAS_CSV,
                 **{f"builder_{k}": v for k, v in BUILDERS.items()}}.items():
    INPUT_HASHES[label] = {"path": str(p.relative_to(REPO)).replace("\\", "/"),
                           "sha256": sha256_file(p), "bytes": p.stat().st_size}

print("INPUT PROVENANCE (SHA-256)")
print("-" * 108)
for label, rec in INPUT_HASHES.items():
    print(f"  {label:18s} {rec['sha256']}  {rec['bytes']:>10,} B  {rec['path']}")

# Compare the data inputs against the digests the original run recorded.
_orig = json.loads((ARCHIVE / "manifest.json").read_text(encoding="utf-8"))
_orig_hashes = {k.replace("\\", "/"): v["sha256"] for k, v in _orig["inputs"].items()}
_by_path = {r["path"]: r["sha256"] for r in INPUT_HASHES.values()}

print("\nCOMPARISON vs the archived 2026-08-02 run (repo was renamed BettingEdgeContinued -> JoSchoAnalytics)")
print("-" * 108)
_drift = []
for path, old in _orig_hashes.items():
    new = _by_path.get(path)
    same = (new == old)
    if not same:
        _drift.append(path)
    print(f"  {'MATCH   ' if same else 'DRIFTED '} {path}")
print(f"\ndata inputs compared: {len(_orig_hashes)} | identical: {len(_orig_hashes) - len(_drift)} | drifted: {len(_drift)}")
assert not _drift, f"source data changed since the archived run: {_drift}"
print("=> every data input is byte-identical to the archived run; recomputed numbers are directly comparable.")

INPUT PROVENANCE (SHA-256)
------------------------------------------------------------------------------------------------------------
  walkforward_RB     c8c0e1584a18452adcb2c3510b1ff91104b8f44e150a0a89b46e21f6a9f04411      49,283 B  fantasy/projections/results/walkforward_predictions.csv
  walkforward_WR     49f6b0d69f796d90c1fe5b3bd9a1fdd0414f36d023a0debe056308ca0d3957db      75,912 B  fantasy/projections/results/wr_walkforward_predictions.csv
  walkforward_TE     ab663974a8888334ee7fd7cf69c893aa5b60cd08e30df31642677e8dee2c8160      39,995 B  fantasy/projections/results/te_walkforward_predictions.csv
  walkforward_QB     b2545b0c55ab3dfbd0ff378b60d60a4ab0e6e16eee283ef30a7e690b2fb33acc      26,699 B  fantasy/projections/results/qb_walkforward_predictions.csv
  season_dataset     dfdf38d9372c830b9fdfffe914024000409e08f01f21536f81876c199e0e2319   2,758,268 B  fantasy/seasonal_projections/season_dataset_2014_2025.csv
  builder_RB         7ffb77d4db746f3ebbc6c2ecb475481b05ef59a46a53063

### Interpretation — the data is byte-identical to the original run

All five data inputs match the digests recorded in `archive/original_2026-08-02/manifest.json`:
**5 compared, 5 identical, 0 drifted**. The repository rename moved files without touching their
contents.

This is the precondition that makes the rest of the notebook meaningful as a *reproduction*. Any
number this run produces that differs from the archived report cannot be blamed on changed data — it
would have to be a change in this notebook's logic, and would need explaining rather than accepting.
As it turns out (Stages 6–10), nothing differs.

The four builder scripts are hashed too but deliberately **not** compared against the archive: the
original manifest recorded only the data inputs, so there is no prior digest to compare against.
Their hashes are recorded here as a baseline for future runs, and the next cell audits their *logic*
directly rather than trusting a hash to stand in for a reading.

`INPUT_HASHES` now holds the provenance baseline. Stage 12b re-hashes all nine at the end of the run,
which is what would catch an analysis that silently rewrote its own inputs.

### Explain — Stage 2: audit the generator, do not trust the CSV

The saved walk-forward CSVs contain *outputs*. The claim that matters for this study — that the
prediction for season *Y* was produced by a model trained only on seasons before *Y* — lives in the
code that wrote them, not in the file. A leaked prediction would make every hit rate below
meaningless, so this cell reads the generating source and shows the guarantee.

**Inputs.** `build_rb_projection.py` (the engine) and the three sibling builders.

**What it extracts and prints.**

1. The body of `walk_forward()` from the RB builder — the function that produced every row in
   every one of the four CSVs. The lines to read are the training slice `df[df.season < Y]` and the
   assertion `assert (tr.season < Y).all()`, which fails the build outright on a leak.
2. Confirmation that `build_{wr,te,qb}_projection.py` each *import* that same `walk_forward` from
   the RB engine rather than defining their own copy — so the guarantee is shared, not duplicated
   four times with three chances to diverge.
3. The `TEST_SEASONS` constant as the builders define it, to confirm the 2021–2025 window is the
   builders' own and not something this notebook imposed after the fact.

**Assumption checked by assertion.** The leak guard must be textually present in the engine, and
all three sibling builders must import the engine. If a future refactor removes either, this cell
fails rather than silently validating a weaker guarantee.

This is a *static* audit — it reads source, it does not re-run training. Re-running the projection
models is explicitly out of scope.

In [3]:
import re

_rb_src = BUILDERS["RB"].read_text(encoding="utf-8")

_m = re.search(r"^def walk_forward\(.*?(?=\n\ndef |\n\n# ---)", _rb_src, re.S | re.M)
assert _m, "walk_forward() not found in the RB engine"
_wf_src = _m.group(0)

print("GENERATOR AUDIT — fantasy/projections/build_rb_projection.py :: walk_forward()")
print("=" * 100)
for i, line in enumerate(_wf_src.splitlines(), 1):
    mark = " <<<" if ("season < Y" in line or "WALK-FORWARD LEAK" in line) else ""
    print(f"  {i:>3} | {line}{mark}")
print("=" * 100)

_train_slice = "df[(df.season < Y)]" in _wf_src
_leak_assert = 'assert (tr.season < Y).all(), f"WALK-FORWARD LEAK' in _wf_src
print(f"training slice restricted to seasons < Y : {_train_slice}")
print(f"explicit leak assertion present          : {_leak_assert}")
assert _train_slice and _leak_assert, "the walk-forward leak guarantee is not present in the engine"

print("\nENGINE REUSE — do the other three positions share this exact function?")
print("-" * 100)
for pos in ("WR", "TE", "QB"):
    src = BUILDERS[pos].read_text(encoding="utf-8")
    imports_engine = bool(re.search(r"from build_rb_projection import\b", src))
    imports_wf = "walk_forward" in re.search(
        r"from build_rb_projection import \(([^)]*)\)", src, re.S).group(1) if imports_engine else False
    defines_own = bool(re.search(r"^def walk_forward\(", src, re.M))
    print(f"  {pos}: imports RB engine={imports_engine} | imports walk_forward={imports_wf} "
          f"| defines its own walk_forward={defines_own}")
    assert imports_engine and imports_wf and not defines_own, f"{pos} does not share the audited engine"

_ts = re.search(r"^TEST_SEASONS = (\[[0-9, ]+\])", _rb_src, re.M).group(1)
print(f"\nbuilders' own TEST_SEASONS constant : {_ts}")
assert eval(_ts) == TEST_SEASONS, "notebook season window disagrees with the builders'"
print("=> every walk-forward row for season Y came from a model fit on seasons < Y only, enforced in code.")

GENERATOR AUDIT — fantasy/projections/build_rb_projection.py :: walk_forward()
    1 | def walk_forward(df, feats, tag):
    2 |     """Per prereg §8: for each Y in 2021-2025, inner-CV select on seasons<Y, fit on seasons<Y, predict Y."""
    3 |     rows, chosen = [], []
    4 |     for Y in TEST_SEASONS:
    5 |         tr = df[(df.season < Y)].dropna(subset=["y"]) <<<
    6 |         te = df[df.season == Y].dropna(subset=["y"])
    7 |         if len(tr) < 60 or len(te) == 0:
    8 |             continue
    9 |         assert (tr.season < Y).all(), f"WALK-FORWARD LEAK ({tag}, {Y})" <<<
   10 |         t0 = time.time()
   11 |         (fam, params, imae), per_family = nested_select(tr, feats)
   12 |         Xtr, Xte = _prep(fam, tr, te, feats)
   13 |         p = _fit_predict(fam, params, Xtr, tr["y"].to_numpy(float), Xte)
   14 |         rows.append(pd.DataFrame({"season": Y, "player_id": te["player_id"].values,
   15 |                                   "player": te["player"].value

### Interpretation — the leak guarantee is in the code, not just in the documentation

The printed `walk_forward()` body settles the central validity question. Line 5 builds the training
frame as `df[(df.season < Y)]` and line 9 asserts `(tr.season < Y).all()` with the message
`WALK-FORWARD LEAK`. So for every test season Y, the model that produced the prediction saw only
strictly earlier seasons, and a violation raises rather than returning a quietly contaminated
number. Both textual checks returned `True`.

The reuse check is what makes this cover all four positions rather than one. WR, TE and QB each
`import` `walk_forward` from the RB engine and **none defines its own** — so there is one
implementation with one guard, not four copies with three opportunities to drift. The builders' own
`TEST_SEASONS` is `[2021, 2022, 2023, 2024, 2025]`, matching this notebook's window exactly, which
confirms the panel was not chosen here after the fact.

What this does **not** establish: it says nothing about whether the *features* are free of
look-ahead (that is the projection pre-registration's problem, not this study's), and it does not
re-run training. It establishes the specific claim this study depends on — that a prediction for
season Y is out-of-sample with respect to season Y.

With the generator trusted, the CSVs can now be loaded as evidence.

### Explain — Stage 3a: load the four walk-forward files and check them individually

Now the data. Each position's walk-forward file is loaded separately and checked *before* anything
is concatenated, so a problem is attributed to the right file.

**Inputs.** The four CSVs hashed in Stage 1b. Schema: `season`, `grp` (`vet`/`rook`), `player_id`,
`player`, `y` (observed season-total half-PPR), `pred` (the model's walk-forward prediction),
`sleeper` (Sleeper's preseason projection as recorded at build time), `model` (the family the
inner CV selected for that fold).

**Per-file assertions.**

- the schema contains every expected column;
- `(season, player_id)` is unique — a duplicated key would double-count a player;
- every season falls inside 2021–2025 — a stray season would mean a file from a different run;
- `y` and `pred` are fully populated (a missing prediction cannot be ranked).

**Output.** One row per position: row count, season coverage, veteran/rookie split, Sleeper
coverage, and the model families the inner CV chose. Then the concatenated frame `WF`, with the
source file recorded in `file_position` so the join in the next cell can verify that the file a row
came from agrees with the position the season dataset assigns it.

**Note on `sleeper`.** This column is taken from the walk-forward file, not the season dataset.
That is deliberate — it is the Sleeper value as of the build, and the next cell quantifies the
difference between the two vintages rather than silently preferring one.

In [4]:
_frames, _rows = [], []
for pos, path in WF_FILES.items():
    d = pd.read_csv(path)
    expected = {"season", "grp", "player_id", "player", "y", "pred", "sleeper", "model"}
    assert expected <= set(d.columns), f"{path.name}: missing columns {expected - set(d.columns)}"
    assert not d.duplicated(["season", "player_id"]).any(), f"{path.name}: duplicate (season, player_id)"
    assert set(d["season"]) <= set(TEST_SEASONS), f"{path.name}: season outside {TEST_SEASONS}"
    assert d["y"].notna().all(), f"{path.name}: null actual"
    assert d["pred"].notna().all(), f"{path.name}: null prediction"
    d["file_position"] = pos
    _frames.append(d)
    _rows.append({
        "position": pos, "rows": len(d),
        "seasons": f"{d.season.min()}-{d.season.max()}", "n_seasons": d.season.nunique(),
        "veteran": int((d.grp == "vet").sum()), "rookie": int((d.grp == "rook").sum()),
        "sleeper_present": int(d.sleeper.notna().sum()),
        "sleeper_cov": f"{d.sleeper.notna().mean():.1%}",
        "model_families": ",".join(sorted(d.model.dropna().unique())),
    })

WF = pd.concat(_frames, ignore_index=True)
assert not WF.duplicated(["season", "player_id", "file_position"]).any()

print("WALK-FORWARD FILES — per-position load and integrity")
print(pd.DataFrame(_rows).to_string(index=False))
print(f"\nconcatenated WF: {len(WF):,} rows x {WF.shape[1]} columns")
print(f"duplicate (season, player_id) across positions: {int(WF.duplicated(['season','player_id']).sum())}")
print("\nrows per season x position:")
print(WF.pivot_table(index="season", columns="file_position", values="player_id", aggfunc="count"))

WALK-FORWARD FILES — per-position load and integrity
position  rows   seasons  n_seasons  veteran  rookie  sleeper_present sleeper_cov      model_families
      RB   802 2021-2025          5      645     157              486       60.6%            lightgbm
      WR  1242 2021-2025          5     1006     236              657       52.9% elasticnet,lightgbm
      TE   677 2021-2025          5      558     119              304       44.9%            lightgbm
      QB   430 2021-2025          5      380      50              262       60.9%    lightgbm,xgboost

concatenated WF: 3,151 rows x 9 columns
duplicate (season, player_id) across positions: 0

rows per season x position:
file_position  QB   RB   TE   WR
season                          
2021           91  169  137  263
2022           90  166  132  254
2023           86  159  134  238
2024           82  157  136  247
2025           81  151  138  240


### Interpretation — 3,151 clean walk-forward rows, and a Sleeper-coverage warning

All four files load with unique `(season, player_id)`, seasons confined to 2021–2025, and no null
predictions or actuals: **802 RB + 1,242 WR + 677 TE + 430 QB = 3,151 rows**, with zero duplicate
keys across positions.

The column that matters most here is **Sleeper coverage, and it is far from complete**: RB 60.6%, WR
52.9%, QB 60.9%, and TE only **44.9%**. Slightly more than half the raw walk-forward population can
express a Sleeper-vs-ADP disagreement at all. That is the entire reason this study needs two rank
universes — universe A leaves those rows in the rank denominators, universe B removes them — and it
is why TE will contribute fewer agreement calls than its row count suggests. (Coverage recovers to
89.2% after the ADP filter in Stage 3c, because Sleeper-less rows are disproportionately players with
no ADP either.)

The `model_families` column is an incidental but useful confirmation that these are genuine nested-CV
outputs: RB and TE selected LightGBM in every fold, while **WR mixed ElasticNet and LightGBM** and
**QB mixed LightGBM and XGBoost**. A single family everywhere would have hinted at a hard-coded
choice; the variation is consistent with the inner CV actually selecting per fold.

Row counts per season rise sharply into 2024–2025 (e.g. WR 238 in 2023 to 247 and 240) — but note
these are pre-ADP-filter counts; the ADP-bearing counts in Stage 3c grow much more steeply, which
shapes the primary panel.

### Explain — Stage 3b: join to the season dataset and prove the join is exact

The walk-forward files carry predictions and outcomes but not the market price. ADP, team, and the
identity fields come from `season_dataset_2014_2025.csv`, joined on `(season, player_id)`.

**Transformation.** A `one_to_one`-validated left join pulling `position`, `team`,
`adp_half_ppr` (the average overall draft pick), `adp_overall_rank`, `adp_pos_rank`,
`sleeper_pts_half_ppr` (the dataset's own Sleeper value), and `is_rookie`.

**Integrity assertions — any failure stops the notebook.**

- the season dataset itself has unique `(season, player_id)`;
- **no walk-forward row fails to join** — an unmatched row would silently vanish from every
  denominator;
- **the joined position equals the source file's position** for every row — this is what makes the
  later "ranks are always within season-position" claim verifiable rather than assumed.

**The Sleeper vintage check.** The dataset was rebuilt on 2026-07-26; the walk-forward files were
written on 2026-07-21. This cell measures the disagreement in three parts: rows where only the
walk-forward file has a Sleeper value, rows where only the dataset has one, and the maximum
absolute difference where both do. The study uses the walk-forward column throughout, so any
dataset-only rows simply drop out of signal evaluation — but the size of that set is measured and
reported rather than assumed negligible.

**Output.** The merged frame `M` plus a `join_diag` dictionary that is carried into the exported
manifest.

In [5]:
SD = pd.read_csv(SEAS_CSV)
assert not SD.duplicated(["season", "player_id"]).any(), "season dataset: duplicate (season, player_id)"

_keep = ["season", "player_id", "position", "team", "adp_half_ppr", "adp_overall_rank",
         "adp_pos_rank", "sleeper_pts_half_ppr", "is_rookie", "norm_name"]
M = WF.merge(SD[_keep], on=["season", "player_id"], how="left", validate="one_to_one")

_unjoined = int(M["position"].isna().sum())
assert _unjoined == 0, f"{_unjoined} walk-forward rows failed the (season, player_id) join"
_mismatch = int((M["position"] != M["file_position"]).sum())
assert _mismatch == 0, f"{_mismatch} rows: joined position disagrees with the source file"

M = M.rename(columns={"position": "pos"}).drop(columns=["file_position"])
M["group"] = np.where(M["grp"] == "rook", "rookie", "veteran")

_both = M.dropna(subset=["sleeper", "sleeper_pts_half_ppr"])
join_diag = {
    "season_dataset_rows": int(len(SD)),
    "walkforward_rows": int(len(WF)),
    "joined": int(len(M)), "unjoined": _unjoined, "position_mismatch": _mismatch,
    "adp_present": int(M.adp_half_ppr.notna().sum()),
    "sleeper_in_walkforward_only": int((M.sleeper.notna() & M.sleeper_pts_half_ppr.isna()).sum()),
    "sleeper_in_dataset_only": int((M.sleeper.isna() & M.sleeper_pts_half_ppr.notna()).sum()),
    "sleeper_value_max_abs_diff": float((_both.sleeper - _both.sleeper_pts_half_ppr).abs().max()),
}

print("JOIN INTEGRITY — walk-forward -> season_dataset_2014_2025.csv on (season, player_id)")
print("-" * 92)
for k, v in join_diag.items():
    print(f"  {k:30s} {v}")

print("\nSLEEPER VINTAGE — walk-forward files (2026-07-21) vs season dataset (rebuilt 2026-07-26)")
print("-" * 92)
_only_ds = M[M.sleeper.isna() & M.sleeper_pts_half_ppr.notna()]
print(f"  rows where ONLY the rebuilt dataset carries a Sleeper value: {len(_only_ds)}")
if len(_only_ds):
    print(_only_ds[["season", "pos", "player", "sleeper_pts_half_ppr", "adp_half_ppr"]]
          .sort_values(["season", "pos"]).to_string(index=False))
print(f"\n  max |difference| where BOTH carry a value: {join_diag['sleeper_value_max_abs_diff']}")
print("  -> this study uses the walk-forward `sleeper` column; the rows above cannot express a")
print("     Sleeper-vs-ADP disagreement and drop out of signal evaluation (they stay in rank denominators).")

JOIN INTEGRITY — walk-forward -> season_dataset_2014_2025.csv on (season, player_id)
--------------------------------------------------------------------------------------------
  season_dataset_rows            7350
  walkforward_rows               3151
  joined                         3151
  unjoined                       0
  position_mismatch              0
  adp_present                    1915
  sleeper_in_walkforward_only    0
  sleeper_in_dataset_only        19
  sleeper_value_max_abs_diff     0.0

SLEEPER VINTAGE — walk-forward files (2026-07-21) vs season dataset (rebuilt 2026-07-26)
--------------------------------------------------------------------------------------------
  rows where ONLY the rebuilt dataset carries a Sleeper value: 19
 season pos           player  sleeper_pts_half_ppr  adp_half_ppr
   2021  RB     Nyheim Hines                  91.1         127.9
   2021  RB Kenneth Gainwell                  97.5         169.4
   2021  WR      Will Fuller                 156

### Interpretation — the join is exact, and the Sleeper vintage gap is 19 rows

All three integrity assertions passed on the full 3,151 rows: **0 unjoined, 0 position mismatches**.
Every walk-forward row found its season-dataset partner, and the position recorded in the source file
agrees with the position the dataset assigns for every single row. The later claim that ranks are
computed within season-position is therefore verified, not assumed.

**1,915 of 3,151 rows carry an ADP** — about 61%. The rest are players the projection models scored
but the market never priced, and they leave the study in the next cell.

The Sleeper vintage difference is smaller and more benign than it could have been. Where both sources
carry a value the **maximum absolute difference is exactly 0.0** — the rebuild did not revise a single
existing projection. The only asymmetry is **19 rows the rebuilt dataset gained**, and the names make
the mechanism obvious: Nyheim Hines, Kenneth Gainwell, Josh Palmer, Will Fuller, Scott Miller, Andrew
Ogletree, Mitchell Tinsley — players whose identity resolution changed between the two builds. There
are **0 rows in the opposite direction**.

19 rows is about 1% of the eligible population, and they are mostly deep (ADP 480–684 for the 2024–25
entries). Using the walk-forward column means those rows sit in the rank denominators but cannot
express a disagreement. Preferring the dataset column would add at most 19 mostly-undrafted rows —
which, given what Stage 6c finds about undrafted rows, would push the headline in the *flattering*
direction. Using the build-time column is the conservative choice and it is the one taken.

### Explain — Stage 3c: apply the population filter, including the QB-rookie exclusion

Two filters define the study population, and both are stated in the brief rather than chosen here.

**1. QB rookies are removed.** The QB rookie arm was *fitted and then held back* from the shipped
surface — `qb_rookie_board_projection.csv` is deliberately header-only — because on 7–13 rookie QBs
a season it mostly re-ranked by draft capital and projected full starter seasons for quarterbacks
the market expected to sit. Scoring a signal the product does not ship would inflate the study with
predictions nobody can act on. RB, WR and TE rookie rows are kept, because those arms *did* ship.

**2. A row must carry an ADP, a prediction, and an actual.** Without a price there is no
disagreement to measure; without an outcome there is nothing to grade.

Note what is deliberately **not** filtered: injuries and games played. Both systems forecast
*season totals*, so availability is part of the quantity being predicted. Removing injured seasons
would grade the projections on a question neither of them was asked.

**Output.** `ELIGIBLE`, plus a per-season-position count table for the whole ADP-bearing
population and for the subset that also carries a Sleeper projection. The gap between those two
tables is exactly the difference between rank universe A and rank universe B in the next stage.

**Assertion.** No QB rookie row may survive — checked explicitly rather than trusted to the filter
expression.

In [6]:
_qb_rookie = M["pos"].eq("QB") & M["grp"].eq("rook")
N_QB_ROOKIE_DROPPED = int(_qb_rookie.sum())

ELIGIBLE = M[~_qb_rookie].copy()
assert not (ELIGIBLE["pos"].eq("QB") & ELIGIBLE["grp"].eq("rook")).any(), "QB rookie row survived the filter"

_pre_adp = len(ELIGIBLE)
ELIGIBLE = ELIGIBLE[ELIGIBLE.adp_half_ppr.notna() & ELIGIBLE.pred.notna() & ELIGIBLE.y.notna()].copy()

print("POPULATION FILTER")
print("-" * 88)
print(f"  joined walk-forward rows                     : {len(M):,}")
print(f"  minus QB rookie rows (arm held from shipping): -{N_QB_ROOKIE_DROPPED}")
print(f"  remaining                                    : {_pre_adp:,}")
print(f"  minus rows with no ADP / pred / actual       : -{_pre_adp - len(ELIGIBLE):,}")
print(f"  ELIGIBLE                                     : {len(ELIGIBLE):,}")

print("\nADP-bearing model population, rows per season x position:")
_a = ELIGIBLE.pivot_table(index="season", columns="pos", values="player_id", aggfunc="count")
_a["TOTAL"] = _a.sum(axis=1)
print(_a)

print("\n... of which also carry a Sleeper projection:")
_b = (ELIGIBLE[ELIGIBLE.sleeper.notna()]
      .pivot_table(index="season", columns="pos", values="player_id", aggfunc="count"))
_b["TOTAL"] = _b.sum(axis=1)
print(_b)

print(f"\nSleeper coverage over the eligible population: "
      f"{ELIGIBLE.sleeper.notna().sum():,}/{len(ELIGIBLE):,} = {ELIGIBLE.sleeper.notna().mean():.1%}")
print(f"veteran / rookie split: {int((ELIGIBLE.group=='veteran').sum()):,} / {int((ELIGIBLE.group=='rookie').sum()):,}")
print(f"drafted subset (adp_overall_rank <= {DRAFTABLE_POOL_SIZE}): "
      f"{int((ELIGIBLE.adp_overall_rank <= DRAFTABLE_POOL_SIZE).sum()):,} rows")

POPULATION FILTER
----------------------------------------------------------------------------------------
  joined walk-forward rows                     : 3,151
  minus QB rookie rows (arm held from shipping): -50
  remaining                                    : 3,101
  minus rows with no ADP / pred / actual       : -1,218
  ELIGIBLE                                     : 1,883

ADP-bearing model population, rows per season x position:
pos     QB   RB   TE   WR  TOTAL
season                          
2021    50   97   44  112    303
2022    46   83   44  106    279
2023    40   92   58  119    309
2024    62  121   82  179    444
2025    68  140  125  215    548

... of which also carry a Sleeper projection:
pos     QB   RB   TE   WR  TOTAL
season                          
2021    38   87   42  103    270
2022    39   79   40  101    259
2023    36   87   51  110    284
2024    58  114   71  162    405
2025    61  119  100  181    461

Sleeper coverage over the eligible population: 1,6

### Interpretation — 1,883 eligible rows, and the drafted subset is less than half

The filter chain is: 3,151 joined rows, minus **50 QB rookie rows**, minus 1,218 rows lacking an ADP,
a prediction or an actual, leaving **1,883 eligible**. The QB-rookie assertion passed, so none
survived.

Two features of the season x position table drive everything downstream.

**The panel is heavily weighted toward recent seasons.** 2021 contributes 303 eligible rows and 2025
contributes **548** — an 81% increase. Most of that is WR (112 to 215) and TE (44 to 125). So the
pooled 2021–2025 panel is not five equal seasons; it is dominated by 2024–2025, and the "primary"
2024–2025 panel is 992 of the 1,883 rows. This is a growth in *ADP coverage* of deep players, not a
growth in the NFL, and it is the mechanism that makes the undrafted tail so influential in the recent
panels specifically.

**Sleeper coverage recovers to 89.2%** (1,679 of 1,883) once the ADP requirement is applied,
substantially better than the 45–61% in the raw files. So universes A and B will differ by only 204
rows, and the sensitivity between them should be modest — which Stage 6b confirms.

**The drafted subset is 886 rows of 1,883 — 47%.** More than half of the "ADP-bearing" population
sits outside the top 180 picks. That single number is the setup for Stage 6c: if the agreement cell
concentrates in that outside-the-draft half, the headline is not measuring what it appears to.

### Explain — Stage 4a: why the stored `adp_pos_rank` column is not used

The season dataset ships an `adp_pos_rank` column. It would be convenient to use it directly. This
cell shows why that would be wrong, because the brief requires the discrepancy to be explained
rather than papered over.

**What the cell does.** It reconstructs a positional ADP rank two ways and compares each against
the stored column:

1. **within the model's walk-forward population** — the rows this study actually evaluates;
2. **within the full ADP-bearing season dataset** — a strictly larger universe.

**What we expect to see.** Neither reconstruction reproduces the stored column. The reason is
visible in `build_season_dataset.py`: `adp_pos_rank` is *merged in verbatim* from an external ADP
source CSV, where it was ranked over that source's own universe — which contains players the season
dataset never retains, and many more that the projection models never scored. Ranks computed over
a superset cannot be reproduced from a subset.

The cell also prints a worked example: a stretch of one season-position where the stored rank skips
a value, which is the visible fingerprint of a player who exists in the ADP source but not here.

**Consequence, and it is a real design decision.** Every rank in this study is rebuilt inside the
stated population. That makes the ranks internally consistent and reproducible, at the cost of not
being the literal market-wide positional rank. It also means the *size* of the population changes
the ranks — which is precisely why the study carries two populations rather than one.

In [7]:
_chk = ELIGIBLE.copy()
_chk["recon_model_pop"] = _chk.groupby(["season", "pos"])["adp_half_ppr"].rank(method="min")

_full = SD[SD.adp_half_ppr.notna() & SD.season.isin(TEST_SEASONS)].copy()
_full["recon_full_dataset"] = _full.groupby(["season", "position"])["adp_half_ppr"].rank(method="min")

adp_rank_diag = {
    "stored_matches_recon_within_model_population": int((_chk.recon_model_pop == _chk.adp_pos_rank).sum()),
    "model_population_rows": int(len(_chk)),
    "stored_matches_recon_within_full_dataset": int((_full.recon_full_dataset == _full.adp_pos_rank).sum()),
    "full_dataset_adp_rows": int(len(_full)),
    "reason": ("adp_pos_rank is merged verbatim from an external ADP source CSV in "
               "build_season_dataset.py and was ranked over that source's universe, a superset of "
               "both the season dataset and the model's scored population; it is not reproducible "
               "from a subset and is therefore NOT used."),
}

print("STORED adp_pos_rank vs RECONSTRUCTION")
print("-" * 92)
print(f"  matches reconstruction within the model population : "
      f"{adp_rank_diag['stored_matches_recon_within_model_population']:,} / {adp_rank_diag['model_population_rows']:,} "
      f"({adp_rank_diag['stored_matches_recon_within_model_population']/adp_rank_diag['model_population_rows']:.1%})")
print(f"  matches reconstruction within the full dataset     : "
      f"{adp_rank_diag['stored_matches_recon_within_full_dataset']:,} / {adp_rank_diag['full_dataset_adp_rows']:,} "
      f"({adp_rank_diag['stored_matches_recon_within_full_dataset']/adp_rank_diag['full_dataset_adp_rows']:.1%})")

print("\nWORKED EXAMPLE — 2025 WR, ordered by ADP, around the first divergence in the full dataset:")
_ex = (_full[(_full.season == 2025) & (_full.position == "WR")]
       .sort_values("adp_half_ppr")[["player", "adp_half_ppr", "adp_overall_rank",
                                     "adp_pos_rank", "recon_full_dataset"]])
_first = _ex.index[(_ex.adp_pos_rank != _ex.recon_full_dataset).to_numpy().argmax()]
_pos_i = _ex.index.get_loc(_first)
print(_ex.iloc[max(0, _pos_i - 3): _pos_i + 4].to_string(index=False))
print("\n  ^ the stored rank jumps while the reconstruction does not: a player priced by the ADP source")
print("    sits in that gap but is absent from this dataset. The stored column counts him; we cannot.")

print("\ngrep of the generating line in build_season_dataset.py:")
_bsd = (SEAS_DIR / "build_season_dataset.py").read_text(encoding="utf-8")
for ln in _bsd.splitlines():
    if "adp_pos_rank" in ln and "keep" in ln:
        print("   ", ln.strip())
print("\n=> DECISION: rebuild every rank inside the stated population; never read adp_pos_rank.")

STORED adp_pos_rank vs RECONSTRUCTION
--------------------------------------------------------------------------------------------
  matches reconstruction within the model population : 917 / 1,883 (48.7%)
  matches reconstruction within the full dataset     : 981 / 1,915 (51.2%)

WORKED EXAMPLE — 2025 WR, ordered by ADP, around the first divergence in the full dataset:
         player  adp_half_ppr  adp_overall_rank  adp_pos_rank  recon_full_dataset
     Josh Downs         119.3             117.0          50.0                50.0
 Jayden Higgins         121.1             119.0          51.0                51.0
 Rashid Shaheed         125.9             123.0          52.0                52.0
 Darnell Mooney         132.0             129.0          54.0                53.0
   Keenan Allen         141.6             131.0          55.0                54.0
Marvin Mims Jr.         145.5             135.0          56.0                55.0
 Christian Kirk         148.5             137.0      

### Interpretation — the stored rank is not reproducible, and the worked example shows why

Neither reconstruction matches the stored column: **917 of 1,883 (48.7%)** within the model
population, **981 of 1,915 (51.2%)** within the full ADP-bearing dataset. Widening the universe
barely helps, which rules out "we simply used too small a pool" as the explanation.

The worked example makes the mechanism concrete. Through 2025 WR, stored rank and reconstruction
agree exactly — Josh Downs 50, Jayden Higgins 51, Rashid Shaheed 52 — and then **Darnell Mooney is
stored as 54 while the reconstruction says 53**, and every later receiver stays one apart. A wide
receiver priced by the ADP source occupies slot 53 and is absent from this dataset entirely. The
stored column counts him; nothing available here can.

The grep confirms it at the source: `adp_pos_rank` appears only in the `keep` list of a merge in
`build_season_dataset.py`. It is copied in, never computed here.

**The consequence is a genuine trade-off, taken deliberately.** Rebuilding ranks in-population makes
every number reproducible from the exported CSVs (Stage 12b verifies exactly that on all 1,298 summary
cells) at the cost of not being the literal market-wide positional rank. It also means **rank values
depend on the population**, which is why `drafted_top180` is a re-ranked population rather than a
filter applied to `all_adp` ranks — and why the two are never mixed in a single table.

### Explain — Stage 4b: build the two rank universes and the two populations

This cell defines `build_ranks()`, the function that turns raw values into the four ranks the whole
study rests on, and applies it across the 2x2 of rank universe x population.

**The four ranks**, all computed **within `(season, position)`** with `method="min"`:

| rank | source | direction | meaning |
|---|---|---|---|
| `adp_rank` | `adp_half_ppr` | ascending | the market's price — rank 1 is the earliest pick |
| `model_rank` | `pred` | descending | our walk-forward projection — rank 1 is the highest projected total |
| `sleeper_rank` | `sleeper` | descending | Sleeper's preseason projection |
| `actual_rank` | `y` | descending | what actually happened |

Ranking within season-position is not a detail — a WR's ADP is only comparable to other WRs in the
same season. A cross-position rank would compare a QB's price to a TE's and make every gap
meaningless. The export in Stage 12 re-derives ranks independently and asserts they match, which is
the check that this never silently breaks.

**Two rank universes.**

- **A — the production-board analogue.** Rank over the whole ADP-bearing model population. A player
  with no Sleeper projection keeps a *missing* Sleeper rank: he still occupies a slot in the ADP,
  model and actual orderings (so he affects everyone else's ranks, exactly as he does on the real
  Draft Board) but he cannot express a disagreement and drops out of signal evaluation.
- **B — the common universe.** Restrict to rows complete on all four quantities *first*, then
  re-rank everything inside that identical set. This is the required sensitivity: it removes the
  asymmetry in A where the denominators include players the signal cannot score.

**Two populations.**

- **`all_adp`** — every eligible row.
- **`drafted_top180`** — `adp_overall_rank <= 180`, the repo's own draftable-universe convention
  (`phase0_benchmark.POOL_SIZE`). In a 12-team, 15-round league roughly 180 players get drafted;
  beyond that, "ADP" is a price nobody actually paid.

**Output.** `RANKED`, a long frame stacking all four combinations with `universe` and `population`
labels, plus a per-combination row count and a completeness table.

In [8]:
def build_ranks(frame: pd.DataFrame, universe: str) -> pd.DataFrame:
    """Rank ADP / model / Sleeper / actual within (season, position) for one rank universe.

    universe 'A' : rank over the full ADP-bearing model population; a missing Sleeper projection
                   yields a missing Sleeper rank (the row still occupies the other three orderings).
    universe 'B' : restrict to rows complete on all four quantities FIRST, then rank inside that set.
    """
    assert universe in ("A", "B"), universe
    d = frame.copy()
    if universe == "B":
        d = d[d["sleeper"].notna()].copy()
    g = d.groupby(["season", "pos"])
    d["adp_rank"] = g["adp_half_ppr"].rank(method="min", ascending=True)
    d["model_rank"] = g["pred"].rank(method="min", ascending=False)
    d["sleeper_rank"] = g["sleeper"].rank(method="min", ascending=False)
    d["actual_rank"] = g["y"].rank(method="min", ascending=False)
    d["universe"] = universe
    d["complete"] = d[["adp_rank", "model_rank", "sleeper_rank", "actual_rank"]].notna().all(axis=1)
    return d


def population_slice(frame: pd.DataFrame, cap):
    return frame if cap is None else frame[frame.adp_overall_rank <= cap]


_stack, _rows = [], []
for pop_name, cap in POPULATIONS.items():
    base = population_slice(ELIGIBLE, cap)
    for uni in ("A", "B"):
        d = build_ranks(base, uni)
        d["population"] = pop_name
        _stack.append(d)
        _rows.append({"population": pop_name, "universe": uni, "rows": len(d),
                      "complete": int(d.complete.sum()),
                      "incomplete": int((~d.complete).sum()),
                      "season_pos_cells": int(d.groupby(["season", "pos"]).ngroups)})
RANKED = pd.concat(_stack, ignore_index=True)

print("RANK CONSTRUCTION — 2 universes x 2 populations")
print(pd.DataFrame(_rows).to_string(index=False))

print("\nCompleteness by population/universe (rows usable for signal evaluation):")
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        d = RANKED[(RANKED.population == pop_name) & (RANKED.universe == uni)]
        print(f"  {pop_name:15s} universe {uni}: {int(d.complete.sum()):>5,} complete of {len(d):>5,}")

print("\nSanity — universes A and B must agree on the COMPLETE row set within a population:")
for pop_name in POPULATIONS:
    a = RANKED[(RANKED.population == pop_name) & (RANKED.universe == "A") & RANKED.complete]
    b = RANKED[(RANKED.population == pop_name) & (RANKED.universe == "B") & RANKED.complete]
    same = set(zip(a.season, a.player_id)) == set(zip(b.season, b.player_id))
    print(f"  {pop_name:15s} identical complete-row membership: {same}  (A={len(a):,}, B={len(b):,})")
    assert same, "universes disagree on which rows are complete"

print("\nRank ranges must start at 1 and never exceed the season-position cell size:")
_bad = 0
for (pop_name, uni, s, p), g in RANKED.groupby(["population", "universe", "season", "pos"]):
    for col in ("adp_rank", "model_rank", "actual_rank"):
        if g[col].min() != 1 or g[col].max() > len(g):
            _bad += 1
print(f"  malformed season-position rank cells: {_bad}")
assert _bad == 0, "rank construction produced an out-of-range rank"

RANK CONSTRUCTION — 2 universes x 2 populations
    population universe  rows  complete  incomplete  season_pos_cells
       all_adp        A  1883      1679         204                20
       all_adp        B  1679      1679           0                20
drafted_top180        A   886       867          19                20
drafted_top180        B   867       867           0                20

Completeness by population/universe (rows usable for signal evaluation):
  all_adp         universe A: 1,679 complete of 1,883
  all_adp         universe B: 1,679 complete of 1,679
  drafted_top180  universe A:   867 complete of   886
  drafted_top180  universe B:   867 complete of   867

Sanity — universes A and B must agree on the COMPLETE row set within a population:
  all_adp         identical complete-row membership: True  (A=1,679, B=1,679)
  drafted_top180  identical complete-row membership: True  (A=867, B=867)

Rank ranges must start at 1 and never exceed the season-position cell size:

### Interpretation — ranks are well-formed and the two universes are consistent

The 2x2 built cleanly. `all_adp` universe A holds 1,883 rows of which **1,679 are complete** (204
lack a Sleeper rank); universe B holds exactly those 1,679. `drafted_top180` A holds 886 rows with
**867 complete** (19 incomplete); B holds 867. All four combinations span the same **20
season-position cells** (5 seasons x 4 positions), so no cell is silently missing.

Two assertions did real work here.

**Complete-row membership is identical between A and B** within each population, verified by set
comparison on `(season, player_id)`. This is the guarantee that A and B are the *same players ranked
differently*, not different samples. Without it, any A-vs-B difference would be confounded by
composition and the sensitivity would be uninterpretable.

**Zero malformed rank cells** across all 80 season-position x universe x population groups: every
rank starts at 1 and none exceeds its cell size. This is the check that catches a `groupby` key
quietly dropping — for example ranking across positions, which would produce maxima far above the
cell size.

Note the incompleteness is much lower on the drafted board (19 of 886, 2.1%) than on the full
population (204 of 1,883, 10.8%). Sleeper simply covers drafted players far better than deep ones,
which is itself a hint that the deep tail is a different kind of data.

`RANKED` now holds four internally consistent orderings per player. Next: turn them into a signal.

### Explain — Stage 5: gaps, agreement, thresholds, the consensus score, and hit/miss/tie

This cell turns ranks into the signal being tested. Every definition here is fixed by the brief.

**The three signed gaps.** Positive means the source ranks the player *above* his draft price:

```
model_gap   = adp_rank - model_rank
sleeper_gap = adp_rank - sleeper_rank
actual_gap  = adp_rank - actual_rank
```

**Agreement at threshold `t`** requires all three of:

```
sign(model_gap) == sign(sleeper_gap)      both point the same way
abs(model_gap)  > t                       strictly greater
abs(sleeper_gap) > t                      strictly greater
```

Ranks are integers, so `>7.5` is exactly "at least 8 spots" and `>0` is "any nonzero
same-direction disagreement".

**The consensus score** takes the *weaker* of the two gaps and carries the shared sign:

```
consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)
```

Using the minimum is a deliberate conservatism: one wildly extreme projection cannot manufacture a
large agreement on its own. Both sources must independently clear the bar.

**Outcome coding.** A hit requires `sign(actual_gap) == sign(consensus_score)` **and**
`actual_gap != 0`. An exact tie (the player finishes at precisely his ADP rank) is scored a
**miss** in the primary rate — the strict reading — with a tie-excluded rate reported alongside so
the choice is visible rather than buried.

**Output.** `SIGNALS`, the full analysis frame with eligibility flags per threshold, plus a worked
example of a single player showing the arithmetic end to end, and the eligible-cell counts.

In [9]:
def add_signals(frame: pd.DataFrame) -> pd.DataFrame:
    d = frame.copy()
    d["model_gap"] = d.adp_rank - d.model_rank
    d["sleeper_gap"] = d.adp_rank - d.sleeper_rank
    d["actual_gap"] = d.adp_rank - d.actual_rank
    d["agree_dir"] = np.sign(d.model_gap) == np.sign(d.sleeper_gap)
    d["consensus_score"] = np.sign(d.model_gap) * np.minimum(d.model_gap.abs(), d.sleeper_gap.abs())
    d["direction"] = np.where(d.consensus_score > 0, "buy",
                              np.where(d.consensus_score < 0, "fade", "none"))
    for t in THRESHOLDS:
        d[thr_col(t)] = (d.complete & d.agree_dir
                         & (d.model_gap.abs() > t) & (d.sleeper_gap.abs() > t)
                         & (d.consensus_score != 0))
    d["outcome"] = np.where(d.actual_gap == 0, "tie",
                            np.where(np.sign(d.actual_gap) == np.sign(d.consensus_score), "hit", "miss"))
    return d


def thr_col(t: float) -> str:
    return "elig_t" + str(t).replace(".", "p")


SIGNALS = add_signals(RANKED)

print("SIGNAL DEFINITIONS APPLIED")
print("-" * 96)
print("  model_gap   = adp_rank - model_rank      (positive: we rank him above his price)")
print("  sleeper_gap = adp_rank - sleeper_rank    (positive: Sleeper ranks him above his price)")
print("  actual_gap  = adp_rank - actual_rank     (positive: he finished above his price)")
print("  consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)   [weaker gap governs]")
print("  hit = sign(actual_gap) == sign(consensus_score) AND actual_gap != 0;  tie counts as a MISS")

_ex = (SIGNALS[(SIGNALS.population == "drafted_top180") & (SIGNALS.universe == "A")
               & (SIGNALS.season == 2024) & SIGNALS[thr_col(10.0)]]
       .nlargest(1, "consensus_score").iloc[0])
print(f"\nWORKED EXAMPLE — {_ex.player} ({_ex.pos}, {int(_ex.season)}), drafted_top180 / universe A")
print(f"  ADP overall {_ex.adp_half_ppr:.1f}  ->  adp_rank        = {int(_ex.adp_rank)}")
print(f"  model pred  {_ex.pred:.1f}  ->  model_rank      = {int(_ex.model_rank)}   model_gap   = {int(_ex.model_gap):+d}")
print(f"  sleeper     {_ex.sleeper:.1f}  ->  sleeper_rank    = {int(_ex.sleeper_rank)}   sleeper_gap = {int(_ex.sleeper_gap):+d}")
print(f"  actual      {_ex.y:.1f}  ->  actual_rank     = {int(_ex.actual_rank)}   actual_gap  = {int(_ex.actual_gap):+d}")
print(f"  agree_dir={_ex.agree_dir}  consensus_score={int(_ex.consensus_score):+d} ({_ex.direction})  -> {_ex.outcome.upper()}")

print("\nELIGIBLE AGREEMENT CELLS by population / universe / threshold (all seasons 2021-2025):")
_tbl = []
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        d = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni) & SIGNALS.complete]
        row = {"population": pop_name, "universe": uni, "complete_rows": len(d)}
        for t in THRESHOLDS:
            row[f"t>{t:g}"] = int(d[thr_col(t)].sum())
        _tbl.append(row)
print(pd.DataFrame(_tbl).to_string(index=False))

print("\nDisagreement direction split at t>0 (all complete rows, both projections nonzero-gapped):")
_d0 = SIGNALS[(SIGNALS.universe == "A") & SIGNALS.complete]
print(_d0.groupby(["population", "direction"]).size().unstack(fill_value=0).to_string())

SIGNAL DEFINITIONS APPLIED
------------------------------------------------------------------------------------------------
  model_gap   = adp_rank - model_rank      (positive: we rank him above his price)
  sleeper_gap = adp_rank - sleeper_rank    (positive: Sleeper ranks him above his price)
  actual_gap  = adp_rank - actual_rank     (positive: he finished above his price)
  consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)   [weaker gap governs]
  hit = sign(actual_gap) == sign(consensus_score) AND actual_gap != 0;  tie counts as a MISS

WORKED EXAMPLE — Michael Wilson (WR, 2024), drafted_top180 / universe A
  ADP overall 201.7  ->  adp_rank        = 73
  model pred  100.4  ->  model_rank      = 55   model_gap   = +18
  sleeper     116.9  ->  sleeper_rank    = 58   sleeper_gap = +15
  actual      101.0  ->  actual_rank     = 49   actual_gap  = +24
  agree_dir=True  consensus_score=+15 (buy)  -> HIT

ELIGIBLE AGREEMENT CELLS by population / universe / threshold (al

### Interpretation — the arithmetic, one player at a time, and the cells it produces

The worked example is the whole study in six lines. **Michael Wilson, WR, 2024**: ADP pick 201.7 makes
him WR73; the model projects him WR55 (`model_gap = +18`); Sleeper projects him WR58
(`sleeper_gap = +15`). Both say underpriced, so `agree_dir` is true and `consensus_score = +15` — the
**weaker** of the two gaps, Sleeper's, not the model's larger +18. He finished WR49, so
`actual_gap = +24` and the call is a hit.

That example also shows why the minimum is the right choice: had the model alone said +18 and Sleeper
+2, the pair would score +2 and fall out of every threshold above 0. Neither source can carry an
agreement by itself.

The eligible-cell table sets expectations for the rest of the notebook, and the two populations behave
very differently:

- `all_adp` universe A: **940 / 516 / 422 / 331** calls at t>0 / >5 / >7.5 / >10 — even the strictest
  threshold keeps 331 calls, 35% of the t>0 cell.
- `drafted_top180` universe A: **410 / 105 / 70 / 37** — the strictest threshold keeps only 9% of its
  t>0 cell, about **7 calls per season across five seasons**.

That contrast is a warning in itself. Large rank gaps are far more available when the rank space runs
to 215 deep receivers than when it stops at the draftable pool. Stage 6c asks whether those extra
calls are signal or noise, and Stage 7 will flag the small drafted cells accordingly.

Direction is close to balanced (`all_adp` 805 buy / 701 fade; drafted 380 / 351), so no result below
is a one-sided artifact of the two systems being uniformly more optimistic than the market.

### Explain — Stage 6a: the summary machinery (Wilson intervals, tie handling, small-cell flags)

Before any results, the scoring functions. Defining them in one place means every table in the
notebook — main, stability, per-position, Underdog — is computed by identical code.

**`wilson(k, n)`** returns a 95% Wilson score interval. Wilson rather than the normal approximation
because several cells here are small and some have hit rates at or near 1.0, where the normal
interval produces nonsense (bounds above 1, or zero width at a boundary). Wilson stays inside
[0, 1] and remains sensible at the extremes.

**`summarise_cell(sub, labels)`** produces one summary row:

- `n`, `hits`, `misses`, `ties`, `hit_rate` (ties in the denominator, counted as misses);
- `wilson_lo` / `wilson_hi`;
- `n_ex_ties` and `hit_rate_ex_ties` — the sensitivity that drops ties entirely;
- `mean_actual_gap` and `median_actual_gap` — how far, in rank spots, the calls actually moved;
- `spearman_consensus_vs_actual_gap` — whether *bigger* agreement corresponds to a *bigger* real
  move, which a hit rate alone cannot show;
- `median_adp_overall` — the median draft price in the cell. This column is included deliberately:
  it is the diagnostic that exposes the artifact in Stage 6c;
- `too_small_n_lt_10` — flagged whenever `n < 10`, marking the cell as carrying no directional
  conclusion.

**Validation.** The cell verifies `wilson()` against three hand-checkable cases and confirms
`summarise_cell` reproduces a hand-computed count on a small slice, so the machinery is proven
before it is trusted.

In [10]:
Z95 = 1.959963984540054


def wilson(k: int, n: int, z: float = Z95):
    """95% Wilson score interval for k successes in n trials; stays within [0, 1]."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centre - half), min(1.0, centre + half))


def summarise_cell(sub: pd.DataFrame, labels: dict) -> dict:
    n = len(sub)
    hits = int((sub.outcome == "hit").sum())
    misses = int((sub.outcome == "miss").sum())
    ties = int((sub.outcome == "tie").sum())
    lo, hi = wilson(hits, n)
    nz = sub[sub.outcome != "tie"]
    rho = (spearmanr(sub.consensus_score, sub.actual_gap).statistic
           if n >= 3 and sub.consensus_score.nunique() > 1 and sub.actual_gap.nunique() > 1
           else np.nan)
    return {**labels, "n": n, "hits": hits, "misses": misses, "ties": ties,
            "hit_rate": hits / n if n else np.nan, "wilson_lo": lo, "wilson_hi": hi,
            "n_ex_ties": len(nz),
            "hit_rate_ex_ties": float((nz.outcome == "hit").mean()) if len(nz) else np.nan,
            "mean_actual_gap": float(sub.actual_gap.mean()) if n else np.nan,
            "median_actual_gap": float(sub.actual_gap.median()) if n else np.nan,
            "spearman_consensus_vs_actual_gap": rho,
            "median_adp_overall": float(sub.adp_half_ppr.median()) if n else np.nan,
            "too_small_n_lt_10": n < 10}


print("VALIDATION — wilson() against hand-checkable cases")
print("-" * 78)
for k, n, note in [(15, 15, "perfect cell: upper bound must be 1.0, lower well below 1"),
                   (0, 15, "zero cell: lower bound must be 0.0"),
                   (50, 100, "balanced: interval must straddle 0.50"),
                   (0, 0, "empty cell: NaN, no crash")]:
    lo, hi = wilson(k, n)
    print(f"  k={k:>3} n={n:>3}  -> [{lo:.4f}, {hi:.4f}]   {note}")
assert wilson(15, 15)[1] == 1.0 and wilson(15, 15)[0] < 1.0
assert wilson(0, 15)[0] == 0.0
_lo, _hi = wilson(50, 100)
assert _lo < 0.5 < _hi
assert all(np.isnan(v) for v in wilson(0, 0))

print("\nVALIDATION — summarise_cell() against a hand-counted slice")
print("-" * 78)
_s = SIGNALS[(SIGNALS.population == "drafted_top180") & (SIGNALS.universe == "A")
             & (SIGNALS.season == 2024) & SIGNALS[thr_col(5.0)]]
_manual = {"n": len(_s), "hits": int((_s.outcome == "hit").sum()),
           "misses": int((_s.outcome == "miss").sum()), "ties": int((_s.outcome == "tie").sum())}
_auto = summarise_cell(_s, {})
print(f"  hand count : {_manual}")
print(f"  summarise  : {{'n': {_auto['n']}, 'hits': {_auto['hits']}, 'misses': {_auto['misses']}, 'ties': {_auto['ties']}}}")
assert all(_auto[k] == v for k, v in _manual.items())
assert _manual["hits"] + _manual["misses"] + _manual["ties"] == _manual["n"], "outcomes must partition the cell"
print(f"  hits + misses + ties == n : True")
print(f"  hit_rate {_auto['hit_rate']:.4f}  Wilson [{_auto['wilson_lo']:.4f}, {_auto['wilson_hi']:.4f}]  "
      f"median ADP {_auto['median_adp_overall']:.1f}  too_small={_auto['too_small_n_lt_10']}")
print("\n=> scoring machinery verified; every table below is computed by these two functions.")

VALIDATION — wilson() against hand-checkable cases
------------------------------------------------------------------------------
  k= 15 n= 15  -> [0.7961, 1.0000]   perfect cell: upper bound must be 1.0, lower well below 1
  k=  0 n= 15  -> [0.0000, 0.2039]   zero cell: lower bound must be 0.0
  k= 50 n=100  -> [0.4038, 0.5962]   balanced: interval must straddle 0.50
  k=  0 n=  0  -> [nan, nan]   empty cell: NaN, no crash

VALIDATION — summarise_cell() against a hand-counted slice
------------------------------------------------------------------------------
  hand count : {'n': 29, 'hits': 25, 'misses': 4, 'ties': 0}
  summarise  : {'n': 29, 'hits': 25, 'misses': 4, 'ties': 0}
  hits + misses + ties == n : True
  hit_rate 0.8621  Wilson [0.6944, 0.9450]  median ADP 115.3  too_small=False

=> scoring machinery verified; every table below is computed by these two functions.


### Interpretation — the scoring functions behave correctly at the boundaries

The Wilson checks confirm the reason for choosing it. A perfect cell `k=15, n=15` returns
**[0.7961, 1.0000]** — the upper bound is 1.0 and the lower bound is a genuine 0.80, whereas the normal
approximation would collapse to zero width at [1.0, 1.0] and imply certainty from fifteen
observations. That case is not hypothetical: `drafted_top180` at `t>10` is exactly 15 for 15 in the
primary panel, so this interval is load-bearing for the headline. The zero cell floors at 0.0, the
balanced cell straddles 0.50, and the empty cell returns NaN without raising.

The `summarise_cell` validation reproduced a hand count exactly — **n=29, hits=25, misses=4, ties=0**
— and `hits + misses + ties == n` confirms the three outcomes partition the cell with nothing
unclassified. The cell also surfaces `median_adp_overall = 115.3` for that slice, the diagnostic column
that Stage 6c depends on.

What this validation does **not** prove: that the *design* is right. It proves the counters count. The
substantive choices — ties as misses, minimum-gap consensus, ranks within season-position — are design
decisions defended in Stage 5, and no passing assertion can vindicate them.

Every table from here on is produced by these two functions, so an error in them would be visible
everywhere at once rather than in one place.

### Explain — Stage 6b: the primary result — pooled 2024-2025, both universes, both populations

The headline table. For the primary panel (2024–2025 pooled), this reports the agreement cell at
each of the four thresholds, for both rank universes and both populations — eight rows per
population, sixteen in total.

**What each column carries.** `n` is the number of agreement calls; `hits`/`misses`/`ties` partition
it; `hit_rate` counts ties as misses; the Wilson interval is the 95% range; `mean`/`median
actual_gap` say how far the calls actually moved in rank spots; `rho` is the Spearman correlation
between agreement strength and the realised move; `median_adp` is the median overall draft pick in
the cell.

**How to read it, and the trap to avoid.** The instinct is to read down the `hit_rate` column and
conclude the signal gets stronger as the threshold rises. Before doing that, read the `median_adp`
column beside it. If a cell's median draft price is deep into the hundreds, those are players nobody
in a real league drafts — and "beating ADP" for a player with no meaningful ADP is not the same
claim as beating the market on a drafted player. Stage 6c pursues that directly.

**Universe A vs B.** These differ only in whether Sleeper-less rows sit in the rank denominators. If
the conclusions move between A and B, the result is population-sensitive and must be reported as
such. If they do not, that axis is settled and the remaining variation is attributable elsewhere.

This cell also builds `SUMMARY`, the full long-format summary across every panel, threshold,
universe, population, and split (overall, direction, position, season, veteran/rookie) — the frame
exported as `threshold_summary.csv` and reconstructed from row-level data in Stage 12.

In [11]:
_summary_rows = []
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        d_all = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni) & SIGNALS.complete]
        for panel, seasons in PANELS.items():
            panel_df = d_all[d_all.season.isin(seasons)]
            for t in THRESHOLDS:
                cell = panel_df[panel_df[thr_col(t)]]
                base = {"market": "sleeper_adp", "population": pop_name, "universe": uni,
                        "panel": panel, "threshold": t, "panel_complete_n": len(panel_df)}
                _summary_rows.append(summarise_cell(cell, {**base, "split": "all", "split_value": "all"}))
                for split_name, col in (("direction", "direction"), ("position", "pos"),
                                        ("season", "season"), ("group", "group")):
                    for val, grp in cell.groupby(col):
                        _summary_rows.append(summarise_cell(
                            grp, {**base, "split": split_name, "split_value": str(val)}))
SUMMARY = pd.DataFrame(_summary_rows)

_SHOW = ["n", "hits", "misses", "ties", "hit_rate", "wilson_lo", "wilson_hi",
         "hit_rate_ex_ties", "mean_actual_gap", "median_actual_gap",
         "spearman_consensus_vs_actual_gap", "median_adp_overall"]


def main_table(panel: str, pop_name: str) -> pd.DataFrame:
    t = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.panel == panel)
                & (SUMMARY.population == pop_name) & (SUMMARY.split == "all")]
    return (t.sort_values(["universe", "threshold"])
             .set_index([t.sort_values(["universe", "threshold"]).universe,
                         t.sort_values(["universe", "threshold"]).threshold])[_SHOW].round(4))


for pop_name, tag in (("all_adp", "FULL ADP-BEARING POPULATION"),
                      ("drafted_top180", f"DRAFTED BOARD (adp_overall_rank <= {DRAFTABLE_POOL_SIZE})")):
    n_complete = int(SUMMARY[(SUMMARY.panel == "pooled_2024_2025") & (SUMMARY.population == pop_name)
                             & (SUMMARY.universe == "A") & (SUMMARY.split == "all")].panel_complete_n.iloc[0])
    print("=" * 118)
    print(f"POOLED 2024-2025 — {tag}   (complete rows in panel: {n_complete:,})")
    print("=" * 118)
    print(main_table("pooled_2024_2025", pop_name).to_string())
    print()

print(f"SUMMARY frame built: {len(SUMMARY):,} rows "
      f"({SUMMARY.split.nunique()} split types, {SUMMARY.panel.nunique()} panels, "
      f"{len(THRESHOLDS)} thresholds, 2 universes, 2 populations)")

POOLED 2024-2025 — FULL ADP-BEARING POPULATION   (complete rows in panel: 866)
                      n  hits  misses  ties  hit_rate  wilson_lo  wilson_hi  hit_rate_ex_ties  mean_actual_gap  median_actual_gap  spearman_consensus_vs_actual_gap  median_adp_overall
universe threshold                                                                                                                                                                     
A        0.0        512   409      97     6    0.7988     0.7619     0.8313            0.8083           9.0938                5.0                            0.7636              459.55
         5.0        338   300      37     1    0.8876     0.8494     0.9170            0.8902          16.1538               14.0                            0.7867              634.25
         7.5        307   277      29     1    0.9023     0.8639     0.9307            0.9052          17.4235               16.0                            0.7878              640.70
 

### Interpretation — a very large headline, and a column that undermines it

**Full ADP-bearing population, universe A, pooled 2024–2025:** the agreement cell hits **79.9%**
(409/512) at t>0, rising monotonically to **88.8%** (338), **90.2%** (307) and **92.3%** (259) at
t>5, t>7.5 and t>10. Wilson intervals are tight and nowhere near 0.50. Spearman between agreement
strength and the realised move runs +0.76 to +0.79, so bigger agreements really do correspond to
bigger moves. Read alone, this looks like a decisive market-beating signal.

**Now read `median_adp_overall` in the same rows: 459.6, 634.3, 640.7, 652.0.** The median agreement
call at `t>5` is a player drafted, on average, at **pick 634**. In a 12-team, 15-round league only
about 180 players are drafted at all. These are not underpriced players; they are players with no
meaningful price. The metric rises with the threshold *and so does the median ADP* — the two move
together, which is exactly the pattern an artifact would produce. Stage 6c tests this directly rather
than leaving it as an impression.

**Drafted board, same panel and universe:** **66.1%** (107/162) at t>0, then **85.0%** (n=40),
**93.1%** (n=29), **100%** (15/15). Median ADP stays at 97–131, inside the draftable range, so these
are calls on players people actually draft. But the sample collapses by a factor of four, and the
Wilson interval at t>10 is **[0.796, 1.000]** — fifteen consecutive correct calls is genuinely
unlikely by chance, and also only fifteen calls.

**Universe A vs B changes nothing.** On the full population B runs 1–3 points lower at every
threshold (77.6 / 85.4 / 87.0 / 88.7); on the drafted board B is within 2 points of A everywhere and
identical at t>10. No conclusion flips. **The rank-universe axis is settled and can be set aside** —
the remaining variation in this study comes from the drafted/undrafted axis and from sample size, not
from how Sleeper-less rows are handled.

Note also that `mean_actual_gap` is much larger in universe A than B on the full population (+9.1 vs
+1.4 at t>0). That is a rank-space effect: A's denominators include the 204 Sleeper-less rows, so the
same player can move more rank spots. It is a reason to compare gap magnitudes only within a universe.

### Explain — Stage 6c: is the full-population result real signal, or the undrafted tail?

The previous cell produced a very high hit rate on the full population. This cell tests the
alternative explanation directly, because a result that large against a market price deserves
suspicion before celebration.

**The hypothesis being tested.** Deep ADP is not a real price. Beyond roughly the 180th pick, the
"average draft position" of a player is an artifact of a handful of drafts in a very long tail — it
carries almost no information about expected production. Two competent projections that both
disagree with a near-random ordering will both be right, and they will agree with each other while
doing it. That produces a high agreement hit rate that has nothing to do with beating a market.

**What the cell shows.**

1. The distribution of overall ADP in the agreement cell versus the whole panel, at each threshold.
   If the cell's ADP distribution shifts progressively deeper as the threshold rises, the "signal
   strengthens with threshold" reading is really "the cell gets more undrafted".
2. The share of each agreement cell that sits **outside** the draftable top 180.
3. Hit rate split by whether the row is inside or outside the top 180 — the direct comparison.
4. The realised season totals of the largest agreement calls in the full population. If the biggest
   "wins" are players who scored a handful of points all year, the cell is ordering noise.

**What would refute the artifact hypothesis.** Roughly equal hit rates inside and outside the top
180, and agreement calls whose median ADP sits inside the drafted range. Anything else confirms
that the full-population headline cannot be quoted as a market-beating result.

In [12]:
print("ADP DEPTH OF THE AGREEMENT CELL — full ADP-bearing population, universe A, pooled 2024-2025")
print("=" * 112)
_panel = SIGNALS[(SIGNALS.population == "all_adp") & (SIGNALS.universe == "A")
                 & SIGNALS.complete & SIGNALS.season.isin([2024, 2025])]
_rows = [{"scope": "whole panel", "n": len(_panel),
          "adp_p25": _panel.adp_half_ppr.quantile(.25), "adp_median": _panel.adp_half_ppr.median(),
          "adp_p75": _panel.adp_half_ppr.quantile(.75),
          "pct_outside_top180": (_panel.adp_overall_rank > DRAFTABLE_POOL_SIZE).mean(),
          "hit_rate": np.nan}]
for t in THRESHOLDS:
    c = _panel[_panel[thr_col(t)]]
    _rows.append({"scope": f"agreement t>{t:g}", "n": len(c),
                  "adp_p25": c.adp_half_ppr.quantile(.25), "adp_median": c.adp_half_ppr.median(),
                  "adp_p75": c.adp_half_ppr.quantile(.75),
                  "pct_outside_top180": (c.adp_overall_rank > DRAFTABLE_POOL_SIZE).mean(),
                  "hit_rate": (c.outcome == "hit").mean()})
print(pd.DataFrame(_rows).round(3).to_string(index=False))

print("\nHIT RATE INSIDE vs OUTSIDE the draftable top 180 (same panel, same agreement cells)")
print("-" * 112)
_split = []
for t in THRESHOLDS:
    c = _panel[_panel[thr_col(t)]].copy()
    c["zone"] = np.where(c.adp_overall_rank <= DRAFTABLE_POOL_SIZE, "inside_top180", "outside_top180")
    for zone, g in c.groupby("zone"):
        lo, hi = wilson(int((g.outcome == "hit").sum()), len(g))
        _split.append({"threshold": t, "zone": zone, "n": len(g),
                       "hits": int((g.outcome == "hit").sum()),
                       "hit_rate": (g.outcome == "hit").mean(), "wilson_lo": lo, "wilson_hi": hi,
                       "median_adp": g.adp_half_ppr.median(),
                       "median_actual_pts": g.y.median()})
print(pd.DataFrame(_split).round(3).to_string(index=False))

print("\nTHE LARGEST FULL-POPULATION AGREEMENT CALLS (t>5) AND WHAT THEY ACTUALLY SCORED")
print("-" * 112)
_big = _panel[_panel[thr_col(5.0)]].nlargest(10, "consensus_score")
print(_big[["season", "pos", "player", "adp_half_ppr", "adp_rank", "model_rank", "sleeper_rank",
            "actual_rank", "consensus_score", "actual_gap", "y", "outcome"]]
      .rename(columns={"y": "actual_half_ppr", "adp_half_ppr": "adp_overall"})
      .round(1).to_string(index=False))
print(f"\n  median season total of those 10 calls: {_big.y.median():.1f} half-PPR points")
print(f"  for scale, the median season total of a top-180 drafted player in this panel: "
      f"{_panel[_panel.adp_overall_rank <= DRAFTABLE_POOL_SIZE].y.median():.1f}")

ADP DEPTH OF THE AGREEMENT CELL — full ADP-bearing population, universe A, pooled 2024-2025
          scope   n  adp_p25  adp_median  adp_p75  pct_outside_top180  hit_rate
    whole panel 866  111.125      258.30  643.875               0.597       NaN
  agreement t>0 512  177.600      459.55  671.625               0.697     0.799
  agreement t>5 338  286.550      634.25  682.200               0.861     0.888
agreement t>7.5 307  296.550      640.70  685.250               0.879     0.902
 agreement t>10 259  412.900      652.00  687.200               0.919     0.923

HIT RATE INSIDE vs OUTSIDE the draftable top 180 (same panel, same agreement cells)
----------------------------------------------------------------------------------------------------------------
 threshold           zone   n  hits  hit_rate  wilson_lo  wilson_hi  median_adp  median_actual_pts
       0.0  inside_top180 155   103     0.665      0.587      0.734       95.40             120.28
       0.0 outside_top180 357   

### Interpretation — confirmed: the full-population headline is an undrafted-tail artifact

The first table settles it. The whole 2024–2025 panel has a median ADP of 258 and is 59.7% outside the
draftable 180. The **agreement cells are progressively more undrafted than the panel they are drawn
from**: 69.7% outside at t>0, then **86.1%, 87.9% and 91.9%** at t>5, t>7.5 and t>10. Median ADP
climbs from 459.6 to 652.0 across the same range. The apparent "signal strengthens with threshold" is,
to a first approximation, "the cell becomes almost entirely undrafted".

The inside/outside split shows where the effect actually lives:

| t | inside top-180 | outside top-180 |
|---|---|---|
| 0 | **66.5%** (n=155) | **85.7%** (n=357) |
| >5 | 85.1% (n=47) | 89.3% (n=291) |
| >7.5 | 89.2% (n=37) | 90.4% (n=270) |
| >10 | 95.2% (n=21) | 92.0% (n=238) |

**This is more nuanced than a blanket dismissal, and both halves must be stated.** At t>0 the gap is
enormous — 19 points — and the pooled 79.9% is essentially a weighted average dragged upward by 357
undrafted calls. At the higher thresholds the two zones converge, and at t>10 the inside rate is
actually *higher*. So the artifact is not that undrafted calls are easy at every threshold; it is that
**the pooled full-population number is dominated by a subpopulation that makes up 86–92% of the cell
and answers a different question**. Quoting 92.3% as a market-beating result remains wrong, because
the number describes a cell that is nine-tenths players nobody drafts.

The realised-points column removes any remaining doubt about what "correct" means out there: median
season total **25.0 half-PPR points outside** the top 180 versus **120.3 inside**. And the ten largest
full-population calls — Tutu Atwell (28.2 points), Nick Westbrook-Ikhine (14.4), Tai Felton (4.0),
Allen Lazard (18.0) — have a median season total of **57.2** against **136.4** for a typical drafted
player. Both projections correctly ordered the noise floor. That is a true statement about deep ADP
being uninformative, not a claim about beating a market.

**Consequence for the rest of the notebook:** the drafted board is the decision-relevant population.
The full population stays in every table for completeness and because the contrast is itself the
finding, but no headline is taken from it.

### Explain — Stage 7: stability across panels, seasons, positions and player group

A result that holds only on the window it was measured on is not a result. This cell widens the
lens in four directions, all on the drafted board and the full population side by side.

**1. Pooled panels.** 2024–2025 (primary), 2023–2025, 2021–2025. The five-season panel is the most
informative because it has the most calls and the least scope for a lucky window; the two-season
panel is a subset of it, not independent evidence.

**2. Per season.** The same agreement cell computed one season at a time. This is where a signal
that is really a two-good-years artifact shows itself. Watch the Wilson intervals: any interval
containing 0.50 is a season the signal did not demonstrably beat a coin flip.

**3. Per position.** QB, RB, TE, WR separately. Positions differ enormously in how many players
carry a meaningful ADP and how noisy season totals are.

**4. Veteran vs rookie.** Rookies have no NFL prior, so the model scores them from an entirely
different feature set.

**Small-cell discipline.** Every cell with `n < 10` is flagged `TOO_SMALL` in the output and carries
no directional conclusion — not a hedge but a hard rule, applied by the `too_small_n_lt_10` column
computed in Stage 6a. On the drafted board, the high thresholds produce single-digit per-season
cells, and those flags will be the majority of that table.

In [13]:
def show_split(split: str, panel: str, pop_name: str, thresholds=THRESHOLDS, universe="A"):
    t = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.panel == panel)
                & (SUMMARY.population == pop_name) & (SUMMARY.universe == universe)
                & (SUMMARY.split == split) & (SUMMARY.threshold.isin(thresholds))]
    cols = ["threshold", "split_value", "n", "hits", "misses", "ties", "hit_rate",
            "wilson_lo", "wilson_hi", "too_small_n_lt_10"]
    out = t.sort_values(["threshold", "split_value"])[cols].round(4).copy()
    out["flag"] = np.where(out.too_small_n_lt_10, "TOO_SMALL", "")
    return out.drop(columns=["too_small_n_lt_10"])


print("=" * 112)
print("1. POOLED PANELS — agreement hit rate (universe A)")
print("=" * 112)
_p = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.universe == "A")
             & (SUMMARY.split == "all") & SUMMARY.panel.isin(POOLED_PANELS)]
_piv = _p.pivot_table(index=["population", "panel"], columns="threshold",
                      values=["hit_rate", "n"], aggfunc="first")
print(_piv.round(4).to_string())

print("\n" + "=" * 112)
print("2. PER SEASON — drafted board, universe A")
print("=" * 112)
for t in THRESHOLDS:
    sub = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.population == "drafted_top180")
                  & (SUMMARY.universe == "A") & (SUMMARY.split == "all")
                  & SUMMARY.panel.str.startswith("season_") & (SUMMARY.threshold == t)]
    sub = sub.sort_values("panel")
    print(f"\n  threshold t>{t:g}")
    for _, r in sub.iterrows():
        flag = "  <-- TOO_SMALL (n<10)" if r.too_small_n_lt_10 else ""
        straddle = "  [interval contains 0.50]" if (r.n >= 10 and r.wilson_lo <= 0.5 <= r.wilson_hi) else ""
        print(f"    {r.panel}: n={int(r.n):>3} hits={int(r.hits):>3} "
              f"hit_rate={r.hit_rate:.4f} [{r.wilson_lo:.3f}, {r.wilson_hi:.3f}]{flag}{straddle}")

print("\n" + "=" * 112)
print("3. PER POSITION — pooled 2024-2025, universe A, t>0 and t>5")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n  -- {pop_name} --")
    print(show_split("position", "pooled_2024_2025", pop_name, [0.0, 5.0]).to_string(index=False))

print("\n" + "=" * 112)
print("4. VETERAN vs ROOKIE and BUY vs FADE — pooled 2024-2025, universe A, t>0 and t>5")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n  -- {pop_name}: veteran/rookie --")
    print(show_split("group", "pooled_2024_2025", pop_name, [0.0, 5.0]).to_string(index=False))
    print(f"  -- {pop_name}: buy/fade --")
    print(show_split("direction", "pooled_2024_2025", pop_name, [0.0, 5.0]).to_string(index=False))

1. POOLED PANELS — agreement hit rate (universe A)
                                hit_rate                            n               
threshold                           0.0     5.0     7.5     10.0 0.0  5.0  7.5  10.0
population     panel                                                                
all_adp        pooled_2021_2025   0.7681  0.8566  0.8815  0.9124  940  516  422  331
               pooled_2023_2025   0.7802  0.8801  0.9024  0.9253  655  392  338  281
               pooled_2024_2025   0.7988  0.8876  0.9023  0.9228  512  338  307  259
drafted_top180 pooled_2021_2025   0.7220  0.8381  0.8857  0.8919  410  105   70   37
               pooled_2023_2025   0.6722  0.8644  0.9250  1.0000  241   59   40   19
               pooled_2024_2025   0.6605  0.8500  0.9310  1.0000  162   40   29   15

2. PER SEASON — drafted board, universe A

  threshold t>0
    season_2021: n= 79 hits= 66 hit_rate=0.8354 [0.739, 0.901]
    season_2022: n= 90 hits= 68 hit_rate=0.7556 [0.658, 0.833

### Interpretation — the drafted-board signal is not stable, and 2025 is its weakest season

**Pooled panels.** On the full population the agreement rate barely moves as the window widens
(76.8 / 78.0 / 79.9% at t>0 for 2021–25 / 2023–25 / 2024–25), which is unsurprising given that the
undrafted tail is present throughout. On the drafted board the pattern **inverts with threshold**: at
t>0 the five-season panel is the *strongest* (72.2%, n=410) and the two-season panel the weakest
(66.1%, n=162), while at t>10 the five-season panel is the weakest (89.2%, n=37) and the short panels
read 100% on 19 and 15 calls. Short panels are not independent evidence — 2024–2025 is a subset of
2023–2025, which is a subset of 2021–2025 — so where they disagree, **the longest panel carries the
most information and the shortest carries the least**.

**Per season is where the honest reading hardens.** Drafted board at t>0: 2021 **83.5%**, 2022 75.6%,
2023 69.6%, 2024 74.4%, and **2025 56.6% with a Wilson interval of [0.454, 0.671] that contains 0.50**.
The most recent complete season is the only one where the threshold-0 signal fails to separate from a
coin flip, and the trend across five seasons is downward rather than flat. A signal that was strongest
in the oldest season and weakest in the newest is the opposite of what a durable edge looks like.

At higher thresholds the per-season cells are simply too thin to read: t>7.5 gives 7, 23, 11, 23 and 6
calls, with 2021 and 2025 flagged `TOO_SMALL` at 100% on 7 and 6 calls; t>10 gives 4, 14, 4, 12 and 3.
Three of five seasons are flagged at t>10 and one of the unflagged ones (2022, 71.4%) has an interval
containing 0.50. **No per-season claim above t>0 is supportable on the drafted board.**

**By position** (drafted, 2024–25, t>0) everything sits between 60% and 73% with wide intervals — QB
60.0% (n=20), TE 60.9% (n=23), WR 62.5% (n=56), RB 73.0% (n=63) — and no position separates from the
others. At t>5 the drafted board yields **2 QB calls, 20 RB, 18 WR and zero TE**, so only RB (90.0%)
has enough calls to look at, and one position-season is no basis for a positional claim.

**Veteran vs rookie** is flat (66.4% vs 63.2% at t>0), so nothing here is driven by the rookie arms.
**Buy vs fade** differs more (70.0% vs 61.1% on the drafted board at t>0), but the intervals overlap
heavily and the ordering reverses at t>5 (81.8% buy vs 88.9% fade). Direction is not a reliable
discriminator at these sample sizes.

### Explain — Stage 8a: the empirical null — does the agreement cell beat chance?

A 66% or 85% hit rate is only impressive relative to what chance would give. The naive reference is
50%, but that is an assumption, not a measurement: ties, direction imbalance, and the specific mix
of season-position cells could all shift the true chance level away from a half.

**The permutation.** `actual_gap` is shuffled **within each `(season, position)` cell**, 10,000
times. Everything else is held fixed — which rows are in the agreement cell, each row's predicted
direction, the thresholds, and the cell sizes. Only the pairing between a player and his realised
outcome is destroyed.

Shuffling within season-position rather than globally is the point: it preserves the actual
distribution of `actual_gap` inside each cell (its spread, its skew, its tie mass), so the null
answers "how often would these *specific* calls be right if outcomes were assigned at random among
these *specific* players?" — not the weaker "how often does a coin land heads?"

**Implementation.** For efficiency, one `(n_perm x N)` int8 matrix of permuted `actual_gap` signs is
built per panel and reused across all four thresholds, so every threshold is tested against the
same null draws. Seeded with `SEED`, so the p-values are reproducible.

**Reported.** Observed hit rate, the null mean, the null 95th percentile, and a one-sided p-value
`(#{null >= observed} + 1) / (n_perm + 1)`. The `+1` makes the p-value conservative and bounds it
away from zero — with 10,000 draws the floor is 1/10001 ≈ 0.0001, and a p at that floor means "not
one of 10,000 random relabelings matched the observed rate", not "p = 0".

**Scope.** This answers *only* whether the cell beats chance. It says nothing about whether our
model adds anything to Sleeper — that is Stage 8b, and conflating the two is the central risk of
this study.

In [14]:
def perm_sign_matrix(d: pd.DataFrame, n_perm: int = N_PERM, seed: int = SEED) -> np.ndarray:
    """(n_perm x len(d)) int8 matrix of sign(actual_gap) after shuffling within (season, position)."""
    rng = np.random.default_rng(seed)
    signs = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    out = np.empty((n_perm, len(d)), dtype=np.int8)
    for idx in d.groupby(["season", "pos"]).indices.values():
        idx = np.asarray(idx)
        order = np.argsort(rng.random((n_perm, len(idx))), axis=1)
        out[:, idx] = signs[idx][order]
    return out


def permutation_test(d: pd.DataFrame, sign_mat: np.ndarray, mask: pd.Series, dir_col: str) -> dict:
    k = int(mask.sum())
    if k == 0:
        return {"n": 0, "observed": np.nan, "null_mean": np.nan, "null_p95": np.nan, "p_value": np.nan}
    dir_sign = np.sign(d[dir_col].to_numpy(float)).astype(np.int8)
    act_sign = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    sel = np.flatnonzero(mask.to_numpy())
    observed = float((act_sign[sel] == dir_sign[sel]).mean())
    null = (sign_mat[:, sel] == dir_sign[sel][None, :]).mean(axis=1)
    return {"n": k, "observed": observed, "null_mean": float(null.mean()),
            "null_p95": float(np.percentile(null, 95)),
            "p_value": float(((null >= observed).sum() + 1) / (len(null) + 1))}


PERM_ROWS, _PANEL_CACHE = [], {}
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        for panel in POOLED_PANELS:
            d = (SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni)
                         & SIGNALS.complete & SIGNALS.season.isin(PANELS[panel])]
                 .reset_index(drop=True))
            sm = perm_sign_matrix(d)
            _PANEL_CACHE[(pop_name, uni, panel)] = d
            for t in THRESHOLDS:
                r = permutation_test(d, sm, d[thr_col(t)], "consensus_score")
                PERM_ROWS.append({"population": pop_name, "universe": uni, "panel": panel,
                                  "threshold": t, "panel_complete_n": len(d), **r})
PERM = pd.DataFrame(PERM_ROWS)

print(f"PERMUTATION NULL — {N_PERM:,} within-(season, position) shuffles of actual_gap, seed={SEED}")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n-- {pop_name}, universe A --")
    t = PERM[(PERM.population == pop_name) & (PERM.universe == "A")]
    print(t.sort_values(["panel", "threshold"])[
        ["panel", "threshold", "n", "observed", "null_mean", "null_p95", "p_value"]
    ].round(4).to_string(index=False))

print("\n" + "-" * 112)
print(f"null mean across ALL {len(PERM)} cells : min {PERM.null_mean.min():.4f}  max {PERM.null_mean.max():.4f}")
print(f"null 95th pct across all cells        : min {PERM.null_p95.min():.4f}  max {PERM.null_p95.max():.4f}")
print(f"p-value across all cells              : min {PERM.p_value.min():.6f}  max {PERM.p_value.max():.6f}")
print(f"p-value resolution floor (1/(n+1))    : {1/(N_PERM+1):.6f}")
print(f"cells with observed <= null 95th pct  : {int((PERM.observed <= PERM.null_p95).sum())} of {len(PERM)}")

PERMUTATION NULL — 10,000 within-(season, position) shuffles of actual_gap, seed=20260802

-- all_adp, universe A --
           panel  threshold   n  observed  null_mean  null_p95  p_value
pooled_2021_2025        0.0 940    0.7681     0.4935    0.5202   0.0001
pooled_2021_2025        5.0 516    0.8566     0.5004    0.5368   0.0001
pooled_2021_2025        7.5 422    0.8815     0.5024    0.5427   0.0001
pooled_2021_2025       10.0 331    0.9124     0.5036    0.5498   0.0001
pooled_2023_2025        0.0 655    0.7802     0.4978    0.5298   0.0001
pooled_2023_2025        5.0 392    0.8801     0.5020    0.5434   0.0001
pooled_2023_2025        7.5 338    0.9024     0.5037    0.5473   0.0001
pooled_2023_2025       10.0 281    0.9253     0.5036    0.5516   0.0001
pooled_2024_2025        0.0 512    0.7988     0.5002    0.5352   0.0001
pooled_2024_2025        5.0 338    0.8876     0.5033    0.5473   0.0001
pooled_2024_2025        7.5 307    0.9023     0.5048    0.5505   0.0001
pooled_2024_2025   

### Interpretation — the cell beats chance decisively, and the 50% reference is empirically correct

Across all **48 permutation cells** the null mean lands between **0.4831 and 0.5050**. That is the
measured answer to a question that had been an assumption: after preserving each season-position
cell's own distribution of outcomes, ties and direction imbalance included, chance really is
approximately a coin flip. The 50% reference used informally elsewhere is legitimate — now verified
rather than asserted.

Every cell returns **p = 0.0001**, the resolution floor for 10,000 draws. That means **not one of
10,000 random relabelings** reached the observed hit rate, in any panel, at any threshold, in either
population. It does not mean p is zero; with this many draws the floor is 1/10,001 and a larger
permutation budget would only push the bound lower.

The strongest single statement is the last line: **0 of 48 cells have an observed rate at or below
their own null 95th percentile.** Even the smallest, most fragile cells clear their own null bar —
`drafted_top180` at t>10 observed 1.0000 against a null 95th percentile of 0.7333 on 15 calls, and
2021–2025 at t>10 observed 0.8919 against 0.6216 on 37. Note how the null p95 widens as cells shrink
(0.5176 at the largest, 0.7333 at the smallest), which is the test correctly demanding more from a
small cell.

**What this does not settle, and it is the point of the next cell.** Beating chance is not the
question Joseph asked. Sleeper's projection alone would also beat chance — comfortably. The
permutation says the agreement cell contains real information; it says nothing about *whose*
information it is. That requires comparing agreement against Sleeper acting alone.

### Explain — Stage 8b: the comparison that actually decides the study

Beating chance is not the question Joseph asked. Sleeper's projection is free and public. The
question is whether *our model's agreement* adds anything to it. This cell isolates that.

**Two comparators, both conditioned on a call already existing.**

1. **Among model calls above the threshold** (`|model_gap| > t`), compare rows where Sleeper agrees
   against rows where Sleeper does not. Both groups are graded against the *model's* direction. This
   asks: does Sleeper's endorsement improve our calls?
2. **Among Sleeper calls above the threshold** (`|sleeper_gap| > t`), compare rows where our model
   agrees against rows where it does not, both graded against *Sleeper's* direction. This asks: does
   our endorsement improve Sleeper's calls? **This is the direction that matters** — it is the only
   one that could justify our model earning a place beside a projection that already exists.

**Uncertainty.** A stratified bootstrap resamples within `(season, position)` 10,000 times,
preserving the panel's composition, and reports a percentile 95% CI on the *difference* in hit
rates. One index matrix per panel is reused across thresholds and both comparators.

**How to read a CI that crosses zero.** It means the data are consistent with no improvement. It is
not proof of no effect, but it is a failure to demonstrate one — and on a post-hoc study with no
pre-registration, failure to demonstrate is where the honest reading has to stop.

**A caution stated up front.** These two systems are **not independent**. Sleeper's projection is
public and our model is trained on overlapping information about the same players. Nothing here
should be read as combining two independent opinions.

In [15]:
def boot_index_matrix(d: pd.DataFrame, n_boot: int = N_BOOT, seed: int = SEED + 1) -> np.ndarray:
    """(n_boot x len(d)) stratified bootstrap index matrix, resampling within (season, position)."""
    rng = np.random.default_rng(seed)
    strata = [np.asarray(ix) for ix in d.groupby(["season", "pos"]).indices.values()]
    return np.concatenate(
        [ix[rng.integers(0, len(ix), size=(n_boot, len(ix)))] for ix in strata], axis=1)


def correct_vec(d: pd.DataFrame, dir_col: str) -> np.ndarray:
    return ((np.sign(d.actual_gap) == np.sign(d[dir_col])) & (d.actual_gap != 0)).to_numpy()


def bootstrap_lift(d, boot_idx, mask_a, mask_b, dir_a, dir_b) -> dict:
    ma, mb = mask_a.to_numpy(), mask_b.to_numpy()
    ca, cb = correct_vec(d, dir_a), correct_vec(d, dir_b)
    if ma.sum() == 0 or mb.sum() == 0:
        return {"hr_agree": np.nan, "hr_no_agree": np.nan, "lift": np.nan,
                "ci_lo": np.nan, "ci_hi": np.nan, "n_agree": int(ma.sum()),
                "n_no_agree": int(mb.sum()), "boot_usable": 0, "ci_crosses_zero": None}
    hr_a, hr_b = float(ca[ma].mean()), float(cb[mb].mean())
    SA, SB = ma[boot_idx], mb[boot_idx]
    na, nb = SA.sum(1), SB.sum(1)
    ok = (na > 0) & (nb > 0)
    lifts = ((ca[boot_idx] & SA).sum(1)[ok] / na[ok]) - ((cb[boot_idx] & SB).sum(1)[ok] / nb[ok])
    lo, hi = float(np.percentile(lifts, 2.5)), float(np.percentile(lifts, 97.5))
    return {"hr_agree": hr_a, "hr_no_agree": hr_b, "lift": hr_a - hr_b, "ci_lo": lo, "ci_hi": hi,
            "n_agree": int(ma.sum()), "n_no_agree": int(mb.sum()), "boot_usable": int(ok.sum()),
            "ci_crosses_zero": bool(lo <= 0.0 <= hi)}


COMP_ROWS = []
for (pop_name, uni, panel), d in _PANEL_CACHE.items():
    bi = boot_index_matrix(d)
    for t in THRESHOLDS:
        both = d[thr_col(t)]
        model_call = (d.model_gap.abs() > t) & (d.model_gap != 0)
        sleeper_call = (d.sleeper_gap.abs() > t) & (d.sleeper_gap != 0)
        base = {"population": pop_name, "universe": uni, "panel": panel, "threshold": t,
                "panel_complete_n": len(d)}
        v_model = bootstrap_lift(d, bi, model_call & both, model_call & ~both, "model_gap", "model_gap")
        v_sleep = bootstrap_lift(d, bi, sleeper_call & both, sleeper_call & ~both, "sleeper_gap", "sleeper_gap")
        COMP_ROWS.append({**base,
                          **{f"vs_model_alone_{k}": v for k, v in v_model.items()},
                          **{f"vs_sleeper_alone_{k}": v for k, v in v_sleep.items()}})
COMPARISONS = pd.DataFrame(COMP_ROWS)

_MC = ["threshold", "vs_model_alone_hr_agree", "vs_model_alone_hr_no_agree", "vs_model_alone_n_no_agree",
       "vs_model_alone_lift", "vs_model_alone_ci_lo", "vs_model_alone_ci_hi", "vs_model_alone_ci_crosses_zero"]
_SC = ["threshold", "vs_sleeper_alone_hr_agree", "vs_sleeper_alone_hr_no_agree", "vs_sleeper_alone_n_no_agree",
       "vs_sleeper_alone_lift", "vs_sleeper_alone_ci_lo", "vs_sleeper_alone_ci_hi", "vs_sleeper_alone_ci_crosses_zero"]

print(f"INCREMENTAL COMPARISONS — stratified bootstrap, {N_BOOT:,} resamples, seed={SEED+1}, universe A")
for pop_name in POPULATIONS:
    for panel in POOLED_PANELS:
        t = COMPARISONS[(COMPARISONS.population == pop_name) & (COMPARISONS.universe == "A")
                        & (COMPARISONS.panel == panel)].sort_values("threshold")
        print("\n" + "=" * 118)
        print(f"{pop_name}  |  {panel}")
        print("=" * 118)
        print("  (1) among MODEL calls: does Sleeper's agreement help OUR calls?")
        print(t[_MC].round(4).to_string(index=False))
        print("\n  (2) among SLEEPER calls: does OUR agreement help SLEEPER's calls?   <-- the decisive direction")
        print(t[_SC].round(4).to_string(index=False))

print("\n" + "=" * 118)
print("VERDICT SCAN — Sleeper-side lift, universe A: does any CI exclude zero?")
print("=" * 118)
_v = COMPARISONS[(COMPARISONS.universe == "A")].sort_values(["population", "panel", "threshold"])
for _, r in _v.iterrows():
    verdict = "CROSSES ZERO -> not demonstrated" if r.vs_sleeper_alone_ci_crosses_zero else "excludes zero"
    print(f"  {r.population:15s} {r.panel:17s} t>{r.threshold:<4g} "
          f"lift {r.vs_sleeper_alone_lift:+.4f} [{r.vs_sleeper_alone_ci_lo:+.4f}, "
          f"{r.vs_sleeper_alone_ci_hi:+.4f}]  {verdict}")

INCREMENTAL COMPARISONS — stratified bootstrap, 10,000 resamples, seed=20260803, universe A

all_adp  |  pooled_2024_2025
  (1) among MODEL calls: does Sleeper's agreement help OUR calls?
 threshold  vs_model_alone_hr_agree  vs_model_alone_hr_no_agree  vs_model_alone_n_no_agree  vs_model_alone_lift  vs_model_alone_ci_lo  vs_model_alone_ci_hi  vs_model_alone_ci_crosses_zero
       0.0                   0.7988                      0.4441                        331               0.3547                0.2913                0.4175                           False
       5.0                   0.8876                      0.4851                        303               0.4024                0.3355                0.4676                           False
       7.5                   0.9023                      0.4834                        271               0.4189                0.3503                0.4870                           False
      10.0                   0.9228                      0.5

### Interpretation — the decisive comparison, and it does not support an incremental claim

**Comparator (1), the weak direction, is unambiguous.** Among our model's own calls, Sleeper's
agreement lifts the hit rate by **+0.21 to +0.48** on the drafted board and +0.34 to +0.42 on the full
population, with every interval clear of zero in all 24 cells. Sleeper's endorsement makes our calls
much better. Given that the shipped models do not beat Sleeper at any position (RB ρ +0.689, WR
+0.736, TE +0.734, QB +0.695 against Sleeper's consistently higher figures), this is close to
expected: conditioning our calls on a better forecaster's agreement should help.

**Comparator (2) is the one that could justify the model earning a place beside Sleeper, and on the
drafted board it fails on the longest panel.** Among Sleeper's calls, adding our agreement gives:

| panel | t>5 | t>7.5 | t>10 |
|---|---|---|---|
| **2021–2025 (five seasons)** | **+0.068 [−0.028, +0.162]** | **+0.073 [−0.028, +0.172]** | **+0.035 [−0.095, +0.156]** |
| 2023–2025 | +0.194 [+0.056, +0.326] | +0.183 [+0.044, +0.317] | +0.216 [+0.091, +0.351] |
| 2024–2025 | +0.153 [−0.004, +0.311] | +0.181 [+0.026, +0.333] | +0.269 [+0.107, +0.450] |

**On the full five-season drafted panel every interval above t>0 crosses zero.** The shorter panels
look positive, and at first glance that reads as a recent-years effect — but they are nested subsets
of the panel that fails, they contain a third to a half as many calls, and Stage 7 showed the
per-season drafted rate *declining* into 2025. The reading that survives all of it is: **on the
drafted board, this study does not demonstrate that our model adds value beyond Sleeper alone.** The
2024–25 t>10 lift of +0.269 rests on 15 agreement calls against 26 non-agreement calls, which is not a
foundation for a claim.

Note the mechanism visible in the `hr_no_agree` column: Sleeper's *unaided* calls on the drafted board
already hit 77.0%, 81.3% and 85.7% at the three thresholds over five seasons. The bar our model has to
clear rises with the threshold, which is precisely why the lift shrinks to +0.035 at t>10 rather than
growing.

**On the full population comparator (2) does clear zero everywhere** (+0.099 to +0.216, all 12 cells).
But Stage 6c established that population is 86–92% undrafted at these thresholds, so what it shows is
that our model helps Sleeper order the noise floor. That is not a draft-board claim.

**Independence is not assumed anywhere.** Sleeper's projection is public and our model trains on
overlapping information about the same players, so these are correlated opinions and no
"two independent signals" framing is available.

### Explain — Stage 9: descriptive logistic — does agreement matter beyond gap size?

The agreement cells are also, mechanically, the cells with the largest gaps: requiring both
`|model_gap| > t` and `|sleeper_gap| > t` selects for magnitude. So part of the agreement effect
might just be "big disagreements are more often right". This cell separates the two.

**Model.** Logistic regression, fit by Newton–Raphson in this notebook (statsmodels is not a
dependency of this environment, and the fit is small enough to do explicitly and transparently):

```
P(model's directional call is correct)
    ~ agree + |model_gap| + |sleeper_gap| + position + season
```

**Read the outcome direction carefully.** The outcome is whether **our model's** call was right, and
`agree` is whether Sleeper pointed the same way. So a positive `agree` coefficient says *Sleeper's
agreement improves our calls*. It does **not** say our model improves Sleeper. That is the same
asymmetry as comparator (1) in Stage 8b, and it is the weaker of the two claims. The decisive
direction remains the Sleeper-side bootstrap.

**Population.** Rows where both projections express a nonzero disagreement and the actual outcome is
not an exact tie, on the 2024–2025 panel, universe B (the common universe, so every row is complete
by construction). Fit separately for both populations.

**Stability reporting.** The fit reports convergence, iteration count, McFadden pseudo-R², and an
explicit separation flag (`|coef| > 10` or an enormous standard error). If a fit is unstable it is
labelled as such rather than presented as a clean coefficient table.

**Status.** Descriptive. Nothing is selected on this model, no product uses it, and it is not a
hypothesis test.

In [16]:
def logistic_newton(X: np.ndarray, y: np.ndarray, names, tol=1e-10, maxit=200) -> dict:
    """Plain Newton-Raphson logistic regression with observed-information standard errors."""
    beta = np.zeros(X.shape[1])
    converged, iters = False, 0
    for iters in range(1, maxit + 1):
        eta = np.clip(X @ beta, -30, 30)
        p = 1.0 / (1.0 + np.exp(-eta))
        W = p * (1 - p)
        H = X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - p))
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            converged = True
            break
    eta = np.clip(X @ beta, -30, 30)
    p = 1.0 / (1.0 + np.exp(-eta))
    W = p * (1 - p)
    cov = np.linalg.pinv(X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1]))
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    z = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
    pval = 2 * (1 - norm.cdf(np.abs(z)))
    ll = float(np.sum(y * np.log(np.clip(p, 1e-12, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-12, 1))))
    pbar = y.mean()
    ll0 = float(len(y) * (pbar * np.log(pbar) + (1 - pbar) * np.log(1 - pbar)))
    return {"names": list(names), "coef": beta.tolist(), "se": se.tolist(), "z": z.tolist(),
            "p": pval.tolist(), "n": int(len(y)), "converged": bool(converged), "iterations": int(iters),
            "pseudo_r2": float(1 - ll / ll0) if ll0 else np.nan,
            "unstable_or_separated": bool(np.max(np.abs(beta)) > 10 or np.nanmax(se) > 50)}


def logistic_design(d: pd.DataFrame):
    cols = [np.ones(len(d))]
    names = ["intercept"]
    cols.append(d.agree_dir.astype(float).to_numpy()); names.append("agree")
    cols.append(d.model_gap.abs().to_numpy(float)); names.append("abs_model_gap")
    cols.append(d.sleeper_gap.abs().to_numpy(float)); names.append("abs_sleeper_gap")
    for pos in sorted(d.pos.unique())[1:]:
        cols.append((d.pos == pos).astype(float).to_numpy()); names.append(f"pos[{pos}]")
    for s in sorted(d.season.unique())[1:]:
        cols.append((d.season == s).astype(float).to_numpy()); names.append(f"season[{s}]")
    return np.column_stack(cols), names


LOGIT = {}
print("DESCRIPTIVE LOGISTIC — outcome = OUR MODEL's directional call is correct")
print("population: universe B, pooled 2024-2025, both gaps nonzero, actual_gap nonzero")
for pop_name in POPULATIONS:
    d = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == "B") & SIGNALS.complete
                & SIGNALS.season.isin([2024, 2025])]
    d = d[(d.model_gap != 0) & (d.sleeper_gap != 0) & (d.actual_gap != 0)]
    y = (np.sign(d.actual_gap) == np.sign(d.model_gap)).astype(float).to_numpy()
    X, names = logistic_design(d)
    fit = logistic_newton(X, y, names)
    fit["baseline_correct_rate"] = float(y.mean())
    fit["agree_share"] = float(d.agree_dir.mean())
    LOGIT[pop_name] = fit
    print("\n" + "=" * 96)
    print(f"{pop_name}: n={fit['n']}  converged={fit['converged']} ({fit['iterations']} iters)  "
          f"pseudo-R2={fit['pseudo_r2']:.4f}  unstable={fit['unstable_or_separated']}")
    print(f"  baseline correct rate {fit['baseline_correct_rate']:.4f} | share with Sleeper agreeing {fit['agree_share']:.4f}")
    print("=" * 96)
    tab = pd.DataFrame({"term": names, "coef": fit["coef"], "se": fit["se"],
                        "z": fit["z"], "p": fit["p"]})
    tab["odds_ratio"] = np.exp(tab.coef)
    print(tab.round(4).to_string(index=False))

print("\nREMINDER: 'agree' measures Sleeper improving OUR calls. It cannot show the reverse.")

DESCRIPTIVE LOGISTIC — outcome = OUR MODEL's directional call is correct
population: universe B, pooled 2024-2025, both gaps nonzero, actual_gap nonzero

all_adp: n=783  converged=True (6 iters)  pseudo-R2=0.1343  unstable=False
  baseline correct rate 0.6501 | share with Sleeper agreeing 0.6424
           term    coef     se       z      p  odds_ratio
      intercept -0.4875 0.2742 -1.7780 0.0754      0.6141
          agree  1.4323 0.1708  8.3850 0.0000      4.1885
  abs_model_gap  0.0251 0.0069  3.6382 0.0003      1.0254
abs_sleeper_gap  0.0066 0.0070  0.9361 0.3492      1.0066
        pos[RB] -0.2735 0.2865 -0.9545 0.3398      0.7607
        pos[TE] -0.2031 0.3077 -0.6600 0.5093      0.8162
        pos[WR] -0.5010 0.2846 -1.7602 0.0784      0.6059
   season[2025]  0.1185 0.1657  0.7150 0.4746      1.1258

drafted_top180: n=289  converged=True (5 iters)  pseudo-R2=0.0736  unstable=False
  baseline correct rate 0.5571 | share with Sleeper agreeing 0.5363
           term    coef     se

### Interpretation — agreement carries information beyond gap size, in the weak direction

Both fits converged cleanly (6 and 5 Newton iterations) with `unstable_or_separated = False`, so the
coefficients are readable.

`agree` is strongly positive in both: **+1.432** (SE 0.171, p < 1e-15, odds ratio **4.19**) on the full
population and **+1.128** (SE 0.253, p ≈ 8e-6, odds ratio **3.09**) on the drafted board. Because
`|model_gap|` and `|sleeper_gap|` are in the same model, this is *not* explained by agreement cells
being the big-gap cells. Tripling the odds that our model's call is correct is a real effect and it
survives controlling for magnitude — which was the question this stage exists to answer.

Magnitude itself contributes much less than one might expect. `abs_model_gap` is positive but tiny
(+0.025 and +0.040 per rank spot; the drafted-board coefficient is only marginally significant at
p = 0.041), and **`abs_sleeper_gap` is insignificant in both fits** (p = 0.35 and p = 0.60) and even
negative on the drafted board. So the informative content is *that Sleeper agrees*, not *how emphatic
Sleeper is* — which is an independent argument for the minimum-gap consensus score used throughout,
since the weaker gap's exact size is not carrying much.

Pseudo-R² is 0.134 and 0.074. Low, and appropriately so: whether a specific player beats his ADP is
mostly unpredictable, and a model claiming otherwise on 289 drafted rows would be the suspicious
result.

**The direction limitation is the whole caveat, and it must not be lost.** The outcome variable is
*our model's* call being correct and `agree` is *Sleeper agreeing with us*. So this says Sleeper
improves us — the same weak direction as comparator (1). **It cannot show that we improve Sleeper**,
because that outcome variable does not appear anywhere in this fit. The decisive evidence remains the
Sleeper-side bootstrap in Stage 8b, and it did not clear zero on the five-season drafted panel.

### Explain — Stage 10: the freshness confound and the dated-market sensitivity

`fantasy/seasonal_projections/PREREGISTRATION.md` records a named confound under **SLEEPER FRESHNESS
ASYMMETRY**: the stored Sleeper projection is a *week-1-eve snapshot*, while Sleeper ADP is a
late-frozen aggregate of drafts held across the whole summer with no timestamp. So part of any
measured edge may be late news — camp injuries, depth-chart decisions — that entered the projection
*after* a share of the ADP sample had already drafted. The prereg states this attaches to **every**
Sleeper-vs-ADP comparison in the repo, including this one.

That makes the primary result a **late-draft, board-analogue signal**, not a clean forecast made
when the market formed.

**The sensitivity.** Replace the untimestamped Sleeper ADP with a *dated* market: the final Underdog
best-ball draft window (W10, or W9 for 2024 where W10 is empty), reconstructed with
`h11_freshness_signal.py`'s own staged, SHA-256-manifested draft dumps. Both sides of the
comparison then reference a market that existed at a known date.

**Constraints honoured.** `verify_manifest()` re-checks all 16 staged files against the recorded
digests before anything is read; the loader touches only local CSVs (no network); nothing is written
back to any research artifact; and the frozen H11/H12 results are not re-fired or altered.

**Why it stays a separate table.** Underdog best ball is a different format from Sleeper half-PPR
redraft — different scoring, different roster construction, therefore different prices. This is a
directional robustness reading, not a replacement result.

**What to look for.** Attenuation against the dated market is the signature of a freshness
component. Complete collapse would mean the effect *is* freshness; unchanged results would mean
freshness contributes nothing. The magnitude of any attenuation is not a measurement of the
freshness share — the cells are too small for that, and no such estimate is attempted.

In [17]:
UNDERDOG = {"status": "not_attempted"}
UD_SUMMARY = pd.DataFrame()
try:
    sys.path.insert(0, str(SEAS_DIR))
    import h11_freshness_signal as H11

    H11.verify_manifest()
    _uframes = []
    for yr in H11.PANEL:
        wadp, counts = H11.load_windows(yr)
        win = H11.FINAL_WIN[yr]
        w = wadp[wadp.grp == win].copy()
        w["season"] = yr
        _uframes.append(w)
        print(f"  {yr}: final window {win:>3}  players priced {len(w):>4}  drafts in window {counts.get(win, 0):>6,}")
    UD = (pd.concat(_uframes, ignore_index=True).rename(columns={"pos_n": "position"})
          .drop_duplicates(["season", "nn", "position"]))

    ud_elig = ELIGIBLE.merge(UD[["season", "nn", "position", "ud_adp"]],
                             left_on=["season", "norm_name", "pos"],
                             right_on=["season", "nn", "position"], how="left")
    _cov_all = float(ud_elig.ud_adp.notna().mean())
    _d180 = ud_elig[ud_elig.adp_overall_rank <= DRAFTABLE_POOL_SIZE]
    _cov_draft = float(_d180.ud_adp.notna().mean())

    ud_elig = ud_elig[ud_elig.ud_adp.notna()].copy()
    ud_elig["adp_half_ppr"] = ud_elig["ud_adp"]      # the dated market replaces the untimed price

    _rows = []
    for pop_name, cap in POPULATIONS.items():
        base = population_slice(ud_elig, cap)
        for uni in ("A", "B"):
            d = add_signals(build_ranks(base, uni))
            d = d[d.complete]
            for panel in POOLED_PANELS:
                p = d[d.season.isin(PANELS[panel])]
                for t in THRESHOLDS:
                    _rows.append(summarise_cell(p[p[thr_col(t)]], {
                        "market": "underdog_final_window", "population": pop_name, "universe": uni,
                        "panel": panel, "threshold": t, "split": "all", "split_value": "all",
                        "panel_complete_n": len(p)}))
    UD_SUMMARY = pd.DataFrame(_rows)
    UNDERDOG = {"status": "reconstructed", "final_window": {str(k): v for k, v in H11.FINAL_WIN.items()},
                "match_rate_all_adp": _cov_all, "match_rate_drafted_top180": _cov_draft,
                "matched_rows": int(len(ud_elig)),
                "manifest_files_verified": 16, "network_used": False}

    print(f"\njoin coverage: all_adp {_cov_all:.1%}  |  drafted_top180 {_cov_draft:.1%} "
          f"({int(_d180.ud_adp.notna().sum())}/{len(_d180)})")

    print("\n" + "=" * 118)
    print("SLEEPER ADP  vs  DATED UNDERDOG FINAL WINDOW — agreement hit rate, universe A")
    print("=" * 118)
    for panel in POOLED_PANELS:
        print(f"\n-- {panel} --")
        rows = []
        for pop_name in POPULATIONS:
            for t in THRESHOLDS:
                s = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.population == pop_name)
                            & (SUMMARY.universe == "A") & (SUMMARY.panel == panel)
                            & (SUMMARY.threshold == t) & (SUMMARY.split == "all")].iloc[0]
                u = UD_SUMMARY[(UD_SUMMARY.population == pop_name) & (UD_SUMMARY.universe == "A")
                               & (UD_SUMMARY.panel == panel) & (UD_SUMMARY.threshold == t)].iloc[0]
                rows.append({"population": pop_name, "threshold": t,
                             "sleeper_n": s.n, "sleeper_hr": s.hit_rate,
                             "underdog_n": u.n, "underdog_hr": u.hit_rate,
                             "delta_hr": u.hit_rate - s.hit_rate,
                             "ud_wilson_lo": u.wilson_lo, "ud_wilson_hi": u.wilson_hi,
                             "ud_too_small": u.too_small_n_lt_10})
        print(pd.DataFrame(rows).round(4).to_string(index=False))
except Exception as exc:
    UNDERDOG = {"status": "BLOCKED", "error": f"{type(exc).__name__}: {exc}"}
    print(f"UNDERDOG CHECK BLOCKED: {UNDERDOG['error']}")

  assert staged-file sha256s match J1 manifest (16/16): PASS


  2021: final window W10  players priced  392  drafts in window  1,826


  2022: final window W10  players priced  463  drafts in window  5,381


  2023: final window W10  players priced  440  drafts in window  5,553


  2024: final window  W9  players priced  446  drafts in window  7,663


  2025: final window W10  players priced  423  drafts in window  3,031

join coverage: all_adp 77.6%  |  drafted_top180 99.5% (882/886)

SLEEPER ADP  vs  DATED UNDERDOG FINAL WINDOW — agreement hit rate, universe A

-- pooled_2024_2025 --
    population  threshold  sleeper_n  sleeper_hr  underdog_n  underdog_hr  delta_hr  ud_wilson_lo  ud_wilson_hi  ud_too_small
       all_adp        0.0        512      0.7988         451       0.7317   -0.0671        0.6890        0.7705         False
       all_adp        5.0        338      0.8876         245       0.8122   -0.0753        0.7587        0.8562         False
       all_adp        7.5        307      0.9023         204       0.8431   -0.0591        0.7869        0.8866         False
       all_adp       10.0        259      0.9228         163       0.8650   -0.0577        0.8041        0.9092         False
drafted_top180        0.0        162      0.6605         172       0.6279   -0.0326        0.5536        0.6966         False
draft

### Interpretation — against a dated market the signal attenuates everywhere but survives

The reconstruction ran offline and clean: **16 of 16 staged files matched their SHA-256 manifest**,
and the final windows carry 1,826 to 7,663 drafts each, pricing 392–463 players per season. Join
coverage is **99.5% on the drafted board** (882 of 886) and 77.6% on the full population — the deep
tail is where Underdog and Sleeper disagree about who is worth pricing at all, which is consistent
with everything Stage 6c found about that region.

**Every single comparison attenuates. There are no exceptions in either population, at any threshold,
in any panel.** On the five-season drafted panel: 72.2% to **68.3%** at t>0, 83.8% to **78.3%** at
t>5, 88.6% to **86.1%** at t>7.5, and 89.2% to **85.7%** at t>10 — deltas of −0.039, −0.055, −0.025
and −0.035. On the full population the deltas are a consistent −0.046 to −0.075.

That uniformity is the informative part. A freshness component that appears in some cells and not
others could be noise; a decline in **all 24 comparisons** is what a systematic timing advantage looks
like. The prereg's SLEEPER FRESHNESS ASYMMETRY is not hypothetical here — the untimestamped Sleeper
ADP is measurably easier to beat than a market with a known date.

**But the signal does not collapse.** At 78.3% and 86.1% on the five-season drafted panel against a
dated market, most of the effect survives. So freshness is a *component*, not the *explanation*.

**How much of a component cannot be measured from this.** The cells are small — the largest drafted
Underdog cell is 129 calls and the 2024–25 t>10 cell is **13 calls**, where the delta reads −0.231 and
means almost nothing. And the format difference is real and uncontrolled: Underdog best ball has
different scoring and roster construction than Sleeper half-PPR redraft, so part of every delta above
is format, not timing. No decomposition is attempted and none should be read into these numbers.

**Consequence for the verdict:** the primary result must be labelled a **late-draft, board-analogue
signal**, not clean forecast skill.

### Explain — Stage 11: player-level audit of the drafted board

Aggregates can hide a cell that is technically correct and practically meaningless. This cell prints
the individual calls so the result can be judged by inspection.

**Scope.** The drafted board (`adp_overall_rank <= 180`), universe A, 2024–2025, agreement at
`t>5`. This is deliberately the *drafted* population: Stage 6c established that the full-population
examples are undrafted players scoring near zero, which are useless as evidence and dangerous as
content.

**Three tables.**

1. **Largest buy hits** — agreement said "above his price", he finished above it.
2. **Largest fade hits** — agreement said "below his price", he finished below it.
3. **Largest misses** by absolute consensus strength — the confident calls that were wrong. These
   matter more than the hits: they show the failure modes.

**Columns.** Both the overall ADP and the positional `adp_rank`, all four ranks, both gaps, the
consensus score, the realised `actual_gap`, and the actual half-PPR season total — so the size of
each call and the size of the realised move can both be seen, and a "hit" that moved two rank spots
is distinguishable from one that moved forty.

**A hard boundary.** These are historical 2024 and 2025 outcomes used to audit a backtest. They are
**not** 2026 recommendations, and this study validates no player-level call. The rows are printed to
show what the aggregate is made of, and any content use has to carry that framing.

In [18]:
AUDIT_COLS = ["season", "pos", "player", "adp_half_ppr", "adp_rank", "model_rank", "sleeper_rank",
              "actual_rank", "model_gap", "sleeper_gap", "consensus_score", "actual_gap",
              "y", "outcome", "group"]
_RENAME = {"adp_half_ppr": "adp_overall", "y": "actual_pts", "pos": "position"}

_aud = SIGNALS[(SIGNALS.population == "drafted_top180") & (SIGNALS.universe == "A")
               & SIGNALS.complete & SIGNALS.season.isin([2024, 2025]) & SIGNALS[thr_col(5.0)]].copy()

print(f"DRAFTED-BOARD AUDIT — universe A, 2024-2025, agreement at t>5   (n = {len(_aud)} calls)")
print(f"outcome split: {dict(_aud.outcome.value_counts())}")
print(f"direction split: {dict(_aud.direction.value_counts())}")

_hits = _aud[_aud.outcome == "hit"]
_miss = _aud[_aud.outcome == "miss"]

print("\n" + "=" * 128)
print("LARGEST BUY HITS  (agreement ranked him above his draft price; he finished above it)")
print("=" * 128)
print(_hits.nlargest(12, "consensus_score")[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "=" * 128)
print("LARGEST FADE HITS  (agreement ranked him below his draft price; he finished below it)")
print("=" * 128)
print(_hits.nsmallest(12, "consensus_score")[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "=" * 128)
print("LARGEST MISSES  (by |consensus_score|) — the confident calls that were wrong")
print("=" * 128)
print(_miss.reindex(_miss.consensus_score.abs().sort_values(ascending=False).index)
      .head(15)[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "-" * 128)
print("How far did the calls actually move, in rank spots?")
print(_aud.groupby("direction").actual_gap.describe()[["count", "mean", "50%", "min", "max"]].round(2).to_string())
print("\nNOTE: 2024/2025 historical outcomes auditing a backtest. NOT 2026 recommendations.")
print("      This study validates no player-level call at any threshold.")

DRAFTED-BOARD AUDIT — universe A, 2024-2025, agreement at t>5   (n = 40 calls)
outcome split: {'hit': np.int64(34), 'miss': np.int64(5), 'tie': np.int64(1)}
direction split: {'buy': np.int64(22), 'fade': np.int64(18)}

LARGEST BUY HITS  (agreement ranked him above his draft price; he finished above it)
 season position            player  adp_overall  adp_rank  model_rank  sleeper_rank  actual_rank  model_gap  sleeper_gap  consensus_score  actual_gap  actual_pts outcome   group
   2024       WR    Michael Wilson        201.7      73.0        55.0          58.0         49.0       18.0         15.0             15.0        24.0       101.0     hit veteran
   2024       RB     Chuba Hubbard        130.6      43.0        29.0          24.0         15.0       14.0         19.0             14.0        28.0       220.1     hit veteran
   2024       WR     Jakobi Meyers        144.5      55.0        29.0          41.0         23.0       26.0         14.0             14.0        32.0       174.5 

### Interpretation — 40 calls in two seasons, and the misses are the familiar failure modes

The full drafted-board `t>5` cell for 2024–2025 is **40 calls: 34 hits, 5 misses, 1 tie**, split 22
buy / 18 fade. That is **20 calls per season** — the practical scale of this signal on a real draft
board, and small enough that a single season's variance moves the hit rate several points.

The **buy hits** are recognisable mid-to-late-round wins: Chuba Hubbard (RB43 by price, finished RB15,
220.1 points), Bucky Irving (RB59 to RB14, 220.9), Rico Dowdle (RB58 to RB17, 196.8), Jakobi Meyers
(WR55 to WR23), Josh Downs (WR66 to WR34). These are exactly the calls a draft board is *for* — cheap
players both projections liked who returned starter value.

The **fade hits** are equally legible: Marquise Brown (WR41 to WR73, 13.6 points), Zamir White (RB24 to
RB55), Deebo Samuel Sr. (WR14 to WR40), Christian Kirk (WR31 to WR62), Anthony Richardson (QB6 to
QB17). Note that a fade hit is not the same as a bust — Deebo still scored 130.1 points; he simply
returned less than his price.

**The five misses are the most useful rows in the notebook.** Chris Godwin Jr. was a buy who suffered a
season-ending ankle injury in week 7. Garrett Wilson was a buy who finished WR48 from WR18. Jahmyr
Gibbs was a *fade* of a player priced RB4 who finished **RB2 with 336.9 points** — the model is
structurally conservative on elite players, a limitation documented in `fantasy/projections/GUIDE.md`
and visible here at the top of the board. Xavier Worthy and Gabe Davis round it out. So the failure
modes are **availability shocks and elite-tier conservatism** — both already-known weaknesses, not new
ones, and neither addressable by tightening a threshold.

The move-size table shows the calls that land do move meaningfully: buys gain a mean of **+17.2 rank
spots** (median +19), fades lose **−13.8** (median −12). But the ranges include a buy that fell 30
spots and a fade that gained 3, so the distribution is wide.

These rows audit a backtest on completed 2024 and 2025 seasons. They are not 2026 recommendations, and
nothing in this study validates any individual call.

### Explain — Stage 12a: build the row-level export and write the machine-readable artifacts

Everything up to here lives in memory. This cell writes the four companion files into `artifacts/`.
They are outputs of this notebook, not a second source of truth — the analysis, the reasoning and
the conclusions live in the notebook itself.

**`player_season_results.csv`** — one row per (population, universe, season, player). Identity, raw
inputs (`adp_half_ppr`, `model_pred`, `sleeper`, `actual_half_ppr`), all four ranks, all three gaps,
the agreement flag, the consensus score and direction, the per-threshold eligibility booleans, the
outcome, the selected model family, and `max_elig_threshold` (the highest threshold at which the row
qualifies) for convenient sorting.

**`threshold_summary.csv`** — every summary cell: both markets (Sleeper ADP and the dated Underdog
window), both populations, both universes, all eight panels, all four thresholds, and all five split
types.

**`incremental_comparisons.csv`** — the permutation nulls joined to the bootstrap lifts, so the
"beats chance" and "adds value beyond Sleeper" columns sit side by side and cannot be confused.

**`manifest.json`** — input paths and SHA-256s, row counts, join diagnostics, the `adp_pos_rank`
explanation, every definition, the exclusions, the logistic fits, the Underdog block, and the run
timestamp.

**Independent rank re-derivation.** Before writing, the export's ranks are recomputed from scratch
by a separate expression and asserted equal to the pipeline's. This is not redundant: it is the
check that catches a `groupby` key silently changing — for instance a rank accidentally taken across
positions instead of within them.

In [19]:
EXPORT_COLS = ["population", "universe", "season", "pos", "player_id", "player", "team", "group",
               "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "pred", "sleeper", "y",
               "adp_rank", "model_rank", "sleeper_rank", "actual_rank",
               "model_gap", "sleeper_gap", "actual_gap", "agree_dir", "consensus_score", "direction",
               "complete"] + [thr_col(t) for t in THRESHOLDS] + ["outcome", "model"]

PLAYER_RESULTS = (SIGNALS[EXPORT_COLS]
                  .rename(columns={"pos": "position", "pred": "model_pred",
                                   "y": "actual_half_ppr", "model": "model_family"})
                  .sort_values(["population", "universe", "season", "position", "adp_rank"])
                  .reset_index(drop=True))
PLAYER_RESULTS["max_elig_threshold"] = np.select(
    [PLAYER_RESULTS[thr_col(t)] for t in sorted(THRESHOLDS, reverse=True)],
    sorted(THRESHOLDS, reverse=True), default=np.nan)

# --- independent re-derivation of every rank, computed a second way ---
_chk = PLAYER_RESULTS.copy()
_g = _chk.groupby(["population", "universe", "season", "position"])
_re = pd.DataFrame({
    "adp_rank": _g["adp_half_ppr"].rank(method="min", ascending=True),
    "model_rank": _g["model_pred"].rank(method="min", ascending=False),
    "sleeper_rank": _g["sleeper"].rank(method="min", ascending=False),
    "actual_rank": _g["actual_half_ppr"].rank(method="min", ascending=False),
})
_mismatch = {c: int((~np.isclose(_re[c], _chk[c], equal_nan=True)).sum()) for c in _re.columns}
print("INDEPENDENT RANK RE-DERIVATION (recomputed within population/universe/season/position)")
for c, n in _mismatch.items():
    print(f"  {c:14s} mismatching rows: {n}")
assert sum(_mismatch.values()) == 0, f"rank re-derivation disagrees: {_mismatch}"

_gap_bad = int((~np.isclose(_chk.adp_rank - _chk.model_rank, _chk.model_gap, equal_nan=True)).sum()
               + (~np.isclose(_chk.adp_rank - _chk.sleeper_rank, _chk.sleeper_gap, equal_nan=True)).sum()
               + (~np.isclose(_chk.adp_rank - _chk.actual_rank, _chk.actual_gap, equal_nan=True)).sum())
print(f"  gap identities (adp_rank - X_rank) mismatching rows: {_gap_bad}")
assert _gap_bad == 0

SUMMARY_OUT = (pd.concat([SUMMARY, UD_SUMMARY], ignore_index=True)
               if len(UD_SUMMARY) else SUMMARY.copy())
COMPARISONS_OUT = COMPARISONS.merge(
    PERM.rename(columns={c: f"perm_{c}" for c in ["n", "observed", "null_mean", "null_p95", "p_value"]}),
    on=["population", "universe", "panel", "threshold", "panel_complete_n"], how="left")

manifest = {
    "study": "Do the current model and Sleeper, agreeing against ADP, pick the right side?",
    "status": "DESCRIPTIVE POST-HOC RESEARCH - not pre-registered, not live-validated",
    "requested": "2026-08-02", "run_timestamp_utc": RUN_TS,
    "active_notebook": str(NOTEBOOK_PATH.relative_to(REPO)).replace("\\", "/"),
    "environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                    "numpy": np.__version__, "pandas": pd.__version__},
    "reproducibility": {"seed": SEED, "n_permutations": N_PERM, "n_bootstrap": N_BOOT},
    "inputs": INPUT_HASHES,
    "definitions": {
        "model_gap": "adp_rank - model_rank", "sleeper_gap": "adp_rank - sleeper_rank",
        "actual_gap": "adp_rank - actual_rank",
        "consensus_score": "sign(model_gap) * min(|model_gap|, |sleeper_gap|)",
        "agreement": "sign(model_gap)==sign(sleeper_gap) AND |model_gap|>t AND |sleeper_gap|>t",
        "thresholds": THRESHOLDS,
        "threshold_note": "strict >; ranks are integers so >7.5 means at least 8 spots",
        "hit": "sign(actual_gap)==sign(consensus_score) and actual_gap != 0; an exact tie counts as a MISS",
        "ranks": "all four ranks computed within (season, position), method='min'",
        "universe_A": "board analogue: rank over the ADP-bearing model population; missing Sleeper -> missing Sleeper rank",
        "universe_B": "common universe: restrict to complete rows first, then re-rank all four",
        "population_all_adp": "every eligible walk-forward row carrying an ADP",
        "population_drafted_top180": f"adp_overall_rank <= {DRAFTABLE_POOL_SIZE} (phase0_benchmark.POOL_SIZE)",
        "outcome": "observed season-total half-PPR; no injury or games-played filter (availability is in the target)",
    },
    "exclusions": {
        "qb_rookie_rows_dropped": N_QB_ROOKIE_DROPPED,
        "qb_rookie_reason": "the QB rookie arm was fitted then held back from the shipped surface",
        "season_2020": "excluded: stored Sleeper artifact is provenance-contaminated (near-actuals)",
    },
    "join_diagnostics": join_diag,
    "adp_pos_rank_diagnostic": adp_rank_diag,
    "row_counts": {"player_season_results": int(len(PLAYER_RESULTS)),
                   "threshold_summary": int(len(SUMMARY_OUT)),
                   "incremental_comparisons": int(len(COMPARISONS_OUT)),
                   "eligible_rows": int(len(ELIGIBLE))},
    "logistic": LOGIT,
    "underdog_secondary": UNDERDOG,
}

PLAYER_RESULTS.to_csv(ARTIFACTS / "player_season_results.csv", index=False)
SUMMARY_OUT.to_csv(ARTIFACTS / "threshold_summary.csv", index=False)
COMPARISONS_OUT.to_csv(ARTIFACTS / "incremental_comparisons.csv", index=False)
(ARTIFACTS / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print(f"\nARTIFACTS WRITTEN -> {ARTIFACTS.relative_to(REPO)}")
for f in sorted(ARTIFACTS.glob("*")):
    print(f"  {f.name:32s} {f.stat().st_size:>10,} B")
print(f"\nplayer_season_results : {len(PLAYER_RESULTS):,} rows x {PLAYER_RESULTS.shape[1]} cols")
print(f"threshold_summary     : {len(SUMMARY_OUT):,} rows "
      f"({dict(SUMMARY_OUT.market.value_counts())})")
print(f"incremental_comparisons: {len(COMPARISONS_OUT):,} rows")

INDEPENDENT RANK RE-DERIVATION (recomputed within population/universe/season/position)
  adp_rank       mismatching rows: 0
  model_rank     mismatching rows: 0
  sleeper_rank   mismatching rows: 0
  actual_rank    mismatching rows: 0
  gap identities (adp_rank - X_rank) mismatching rows: 0



ARTIFACTS WRITTEN -> fantasy\projections\research\adp_consensus_agreement_2026-08-02\artifacts
  incremental_comparisons.csv          17,119 B
  manifest.json                         8,389 B
  player_season_results.csv           983,763 B
  threshold_summary.csv               256,701 B

player_season_results : 5,315 rows x 32 cols
threshold_summary     : 1,346 rows ({'sleeper_adp': np.int64(1298), 'underdog_final_window': np.int64(48)})
incremental_comparisons: 48 rows


### Interpretation — artifacts written, and the ranks survive an independent re-derivation

The re-derivation returned **0 mismatching rows on all four ranks and 0 on all three gap identities**,
computed from the exported frame by a separately-written expression rather than reused from the
pipeline. This is the check that would catch the most damaging silent error available in this study —
a rank taken over the wrong grouping. It did not happen.

Four artifacts written. **`player_season_results.csv`: 5,315 rows x 32 columns.** That is
1,883 + 1,679 + 886 + 867 across the four population/universe combinations, so every row appears once
per combination it belongs to, each carrying its own correctly-scoped ranks. Anyone reading this file
must filter on `population` and `universe` before aggregating — the row count is intentionally not a
player count.

**`threshold_summary.csv`: 1,346 rows** — 1,298 Sleeper-ADP cells plus 48 Underdog cells.
**`incremental_comparisons.csv`: 48 rows**, with the permutation nulls merged onto the bootstrap lifts
so "beats chance" and "adds value beyond Sleeper" sit in the same row and cannot be confused for one
another.

`manifest.json` carries the nine input hashes, both diagnostic blocks, every definition, the
exclusions with their reasons, both logistic fits, and the Underdog block including its offline status
and join coverage.

These are outputs, not a parallel narrative. Every conclusion in this study is in the notebook; the
CSVs let someone recompute them without re-running anything, which the next cell proves is actually
possible.

### Explain — Stage 12b: reconstruct every summary from row-level data and re-hash the inputs

Two reproducibility guarantees, both required by the brief and both enforced by assertion.

**1. Every exported summary cell must be reconstructible from the exported row-level file.** The
notebook re-reads `artifacts/player_season_results.csv` from disk — not the in-memory frame — and
for **every** Sleeper-ADP row of `threshold_summary.csv` independently re-derives the population,
universe, panel, threshold and split filters, then recomputes `n`, `hits`, `misses` and `ties`. Any
disagreement fails.

This is a stronger check than it first appears. It proves the summary was not computed on a
differently-filtered frame than the one exported, that the split labels mean what they say, and that
a reader who only has the CSVs can rebuild every published number. Underdog rows are excluded from
this check for a stated reason: they are computed against a different market whose per-player rows
are not part of this export.

**2. Every input must be byte-identical to its Stage-1b digest.** Re-hashing at the end catches
anything that rewrote a source during the run — the failure mode where an analysis silently
regenerates the data it is measuring.

The cell also re-verifies the eight archived originals against the digests recorded when they were
moved, confirming the archive is a faithful copy of the original run and not a re-derivation.

In [20]:
print("RECONSTRUCTION CHECK — rebuild every summary cell from the exported row-level CSV")
print("=" * 100)
_rl = pd.read_csv(ARTIFACTS / "player_season_results.csv")
_sm = pd.read_csv(ARTIFACTS / "threshold_summary.csv")
_sleeper_rows = _sm[_sm.market == "sleeper_adp"]

_SPLIT_COL = {"direction": "direction", "position": "position", "season": "season", "group": "group"}
_checked = _failed = 0
_failures = []
for _, r in _sleeper_rows.iterrows():
    d = _rl[(_rl.population == r.population) & (_rl.universe == r.universe)
            & _rl.complete & _rl.season.isin(PANELS[r.panel])]
    cell = d[d[thr_col(r.threshold)]]
    if r.split != "all":
        cell = cell[cell[_SPLIT_COL[r.split]].astype(str) == str(r.split_value)]
    got = (len(cell), int((cell.outcome == "hit").sum()),
           int((cell.outcome == "miss").sum()), int((cell.outcome == "tie").sum()))
    want = (int(r.n), int(r.hits), int(r.misses), int(r.ties))
    _checked += 1
    if got != want:
        _failed += 1
        _failures.append((r.population, r.universe, r.panel, r.threshold, r.split, r.split_value, want, got))

print(f"  summary cells checked : {_checked:,}  (every sleeper_adp row)")
print(f"  reconstruction failures: {_failed}")
if _failures:
    for f in _failures[:10]:
        print("   ", f)
assert _failed == 0, f"{_failed} summary cells could not be reconstructed from row-level data"
print(f"  underdog rows excluded by design: {int((_sm.market == 'underdog_final_window').sum())} "
      f"(different market, per-player rows not in this export)")
print("  => every published Sleeper-ADP number is rebuildable from the exported CSVs alone.")

print("\nINPUT RE-HASH — sources must be unchanged since Stage 1b")
print("=" * 100)
_drift = []
for label, rec in INPUT_HASHES.items():
    now = sha256_file(REPO / rec["path"])
    ok = now == rec["sha256"]
    if not ok:
        _drift.append(label)
    print(f"  {'UNCHANGED' if ok else 'DRIFTED  '}  {label:18s} {now[:24]}...")
assert not _drift, f"source artifacts changed during the run: {_drift}"

print("\nARCHIVE INTEGRITY — the original 2026-08-02 run, verified against its move receipt")
print("=" * 100)
_receipt = json.loads((ARCHIVE / "archive_move_receipt.json").read_text(encoding="utf-8"))
_bad = []
for name, rec in _receipt.items():
    now = sha256_file(ARCHIVE / name)
    ok = now == rec["sha256_after"] == rec["sha256_before"]
    if not ok:
        _bad.append(name)
    print(f"  {'OK      ' if ok else 'MISMATCH'}  {name:36s} {now[:24]}...  {rec['bytes']:>9,} B")
assert not _bad, f"archived originals altered: {_bad}"
print(f"\n  {len(_receipt)} archived files, all byte-identical to their pre-move state.")

RECONSTRUCTION CHECK — rebuild every summary cell from the exported row-level CSV


  summary cells checked : 1,298  (every sleeper_adp row)
  reconstruction failures: 0
  underdog rows excluded by design: 48 (different market, per-player rows not in this export)
  => every published Sleeper-ADP number is rebuildable from the exported CSVs alone.

INPUT RE-HASH — sources must be unchanged since Stage 1b
  UNCHANGED  walkforward_RB     c8c0e1584a18452adcb2c351...
  UNCHANGED  walkforward_WR     49f6b0d69f796d90c1fe5b3b...
  UNCHANGED  walkforward_TE     ab663974a8888334ee7fd7cf...
  UNCHANGED  walkforward_QB     b2545b0c55ab3dfbd0ff378b...
  UNCHANGED  season_dataset     dfdf38d9372c830b9fdfffe9...
  UNCHANGED  builder_RB         7ffb77d4db746f3ebbc6c2ec...
  UNCHANGED  builder_WR         bd29dc1856be12d135d0e195...
  UNCHANGED  builder_TE         71f8278e9014de52b2d410da...
  UNCHANGED  builder_QB         fae87aa98030558f15441deb...

ARCHIVE INTEGRITY — the original 2026-08-02 run, verified against its move receipt
  OK        adp_consensus_experiment.ipynb       ac93

### Interpretation — every published number is rebuildable, and nothing moved during the run

**All 1,298 Sleeper-ADP summary cells reconstructed from the exported row-level CSV with 0 failures.**
The check re-read `player_season_results.csv` from disk and independently re-derived each cell's
population, universe, panel, threshold and split filters before recounting. That the counts agree
everywhere proves three things at once: the summary was computed on the frame that was exported; the
split labels mean what they claim; and a reader holding only the CSVs can rebuild every published
number without this notebook. The 48 Underdog rows are excluded for the stated reason — they are
computed against a different market whose per-player rows are not in this export — and that exclusion
is reported rather than silently applied.

**All nine inputs re-hashed UNCHANGED.** Nothing this notebook did rewrote a source. Combined with
Stage 1b's match against the archived manifest, the inputs were identical to the original run when
read, and identical again when the run finished.

**All eight archived originals verified byte-identical** to the digests captured at the moment they
were moved — the two handoffs, the original notebook, `REPORT.md`, and the four original artifacts.
The archive is a faithful copy of the 2026-08-02 run, not a re-derivation of it, so the comparison
between then and now is a real comparison.

On that comparison: every headline figure recomputed to the archived value exactly — 79.9% / 88.8% /
90.2% / 92.3% on the full population and 66.1% / 85.0% / 93.1% / 100% on the drafted board for the
primary panel, with the same permutation p-values and the same bootstrap intervals. **There is no
discrepancy to resolve.** The one difference between the two runs is presentational: the original
`threshold_summary.csv` and this one both hold 1,346 rows, and the archived `REPORT.md` cited that
figure after a correction, which this notebook now computes and prints directly rather than
transcribing.

### Explain — Stage 12c: the notebook audits its own structure

The last check is the notebook inspecting itself. It parses `adp_consensus_pipeline.ipynb` as JSON
from disk and enforces every structural rule the brief and `memory/prefer-ipynb-not-py.md` require:

- the first cell is markdown beginning `# Introduction`;
- the last cell is markdown beginning `# Conclusion and Next Steps`;
- every code cell's immediate predecessor is markdown whose heading begins `### Explain`;
- every code cell's immediate successor is markdown whose heading begins `### Interpretation`;
- no two code cells are adjacent;
- every code cell carries an execution count;
- no code cell contains an error output or a traceback;
- no Interpretation cell contains prospective placeholder language ("this should show", "we
  expect") — interpretations must describe observed output;
- no companion `.py` analysis or validation script has been left in the project folder.

**How the execution-count and error checks are possible from inside the run.** The notebook is
executed twice. Pass 1 produces a fully executed file on disk; pass 2 re-executes that file, so when
this cell reads `NOTEBOOK_PATH` it is reading a *completed* notebook with all execution counts and
outputs populated. Both passes are structurally identical and deterministic (fixed seeds), so the
audit describes the artifact that ships. The cell prints which state it read so this is explicit
rather than implied.

A `PENDING` result on the execution-count line means this is pass 1; the shipped notebook shows the
pass-2 result, where every check is decided.

In [21]:
print("STRUCTURAL SELF-AUDIT —", NOTEBOOK_PATH.name)
print("=" * 104)
_nb = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
_cells = _nb["cells"]
_md = [c for c in _cells if c["cell_type"] == "markdown"]
_code = [c for c in _cells if c["cell_type"] == "code"]


def _src(c):
    return "".join(c["source"])


_prior_run = all(c.get("execution_count") is not None for c in _code)
print(f"cells: {len(_cells)} total  |  {len(_md)} markdown  |  {len(_code)} code")
print(f"file state read from disk: {'EXECUTED (pass 2 audit — final)' if _prior_run else 'UNEXECUTED (pass 1 — execution checks PENDING)'}")
print("-" * 104)

checks = []
checks.append(("first cell is markdown '# Introduction'",
               _cells[0]["cell_type"] == "markdown" and _src(_cells[0]).lstrip().startswith("# Introduction")))
checks.append(("last cell is markdown '# Conclusion and Next Steps'",
               _cells[-1]["cell_type"] == "markdown"
               and _src(_cells[-1]).lstrip().startswith("# Conclusion and Next Steps")))

_bad_prev, _bad_next, _adjacent = [], [], []
for i, c in enumerate(_cells):
    if c["cell_type"] != "code":
        continue
    prev, nxt = (_cells[i - 1] if i else None), (_cells[i + 1] if i + 1 < len(_cells) else None)
    if not (prev and prev["cell_type"] == "markdown" and _src(prev).lstrip().startswith("### Explain")):
        _bad_prev.append(i)
    if not (nxt and nxt["cell_type"] == "markdown" and _src(nxt).lstrip().startswith("### Interpretation")):
        _bad_next.append(i)
    if prev is not None and prev["cell_type"] == "code":
        _adjacent.append(i)
checks.append(("every code cell preceded by '### Explain' markdown", not _bad_prev))
checks.append(("every code cell followed by '### Interpretation' markdown", not _bad_next))
checks.append(("no two code cells adjacent", not _adjacent))

_no_exec = [i for i, c in enumerate(_cells) if c["cell_type"] == "code" and c.get("execution_count") is None]
_seq = [c.get("execution_count") for c in _code]
checks.append(("every code cell has an execution count",
               (not _no_exec) if _prior_run else None))
checks.append(("execution counts are sequential 1..N",
               _seq == list(range(1, len(_code) + 1)) if _prior_run else None))

_self_idx = max(i for i, c in enumerate(_cells) if c["cell_type"] == "code")   # this very cell
_errs, _self_err = [], False
for i, c in enumerate(_cells):
    if c["cell_type"] != "code":
        continue
    bad = any(o.get("output_type") == "error" or "traceback" in o for o in c.get("outputs", []))
    if bad and i == _self_idx:
        _self_err = True          # this cell's own failure on a PRIOR pass; excluded to avoid a fixed point
    elif bad:
        _errs.append(i)
checks.append(("no code cell contains an error output / traceback (excl. this audit cell)", not _errs))
checks.append(("this audit cell did not itself fail on the previous pass", not _self_err))

_with_out = [c for c in _code if c.get("outputs")]
checks.append(("every code cell has stored output",
               len(_with_out) == len(_code) if _prior_run else None))

_placeholders = ["this " + "should show", "we " + "expect", "should " + "produce",
                 "PLACE" + "HOLDER", "TO" + "DO"]   # split so this line is not its own tripwire
_interp = [c for c in _md if _src(c).lstrip().startswith("### Interpretation")]
_stale = [i for i, c in enumerate(_interp)
          if any(p.lower() in _src(c).lower() for p in _placeholders)]
checks.append((f"no prospective placeholder language in {len(_interp)} Interpretation cells", not _stale))

_stray = sorted(p.name for p in PROJECT.glob("*.py"))
checks.append(("no companion .py analysis/validation script in the project folder", not _stray))

_expected_artifacts = {"threshold_summary.csv", "player_season_results.csv",
                       "incremental_comparisons.csv", "manifest.json"}
checks.append(("all four artifacts present",
               _expected_artifacts <= {p.name for p in ARTIFACTS.glob("*")}))

for name, ok in checks:
    label = "PASS" if ok is True else ("PENDING (pass 1)" if ok is None else "FAIL")
    print(f"  [{label:>16}] {name}")
    if ok is False:
        for lbl, bad in (("no Explain above", _bad_prev), ("no Interpretation below", _bad_next),
                         ("adjacent code", _adjacent), ("no exec count", _no_exec),
                         ("error output", _errs), ("stray .py", _stray)):
            if bad:
                print(f"        -> {lbl}: {bad}")

_decided = [ok for _, ok in checks if ok is not None]
print("-" * 104)
print(f"STRUCTURAL AUDIT: {sum(bool(o) for o in _decided)}/{len(_decided)} decided checks PASS"
      f"{'' if _prior_run else f'  ({sum(1 for _, o in checks if o is None)} pending until pass 2)'}")
assert all(_decided), "structural audit failed"
STRUCTURAL_AUDIT = {"cells_total": len(_cells), "markdown": len(_md), "code": len(_code),
                    "file_state": "executed" if _prior_run else "unexecuted",
                    "checks": {n: ("pass" if o is True else "pending" if o is None else "fail")
                               for n, o in checks}}
print("=" * 104)

STRUCTURAL SELF-AUDIT — adp_consensus_pipeline.ipynb
cells: 65 total  |  44 markdown  |  21 code
file state read from disk: EXECUTED (pass 2 audit — final)
--------------------------------------------------------------------------------------------------------
  [            PASS] first cell is markdown '# Introduction'
  [            PASS] last cell is markdown '# Conclusion and Next Steps'
  [            PASS] every code cell preceded by '### Explain' markdown
  [            PASS] every code cell followed by '### Interpretation' markdown
  [            PASS] no two code cells adjacent
  [            PASS] every code cell has an execution count
  [            PASS] execution counts are sequential 1..N
  [            PASS] no code cell contains an error output / traceback (excl. this audit cell)
  [            PASS] this audit cell did not itself fail on the previous pass
  [            PASS] every code cell has stored output
  [            PASS] no prospective placeholder language in 

### Interpretation — the notebook satisfies its own structural contract

The audit reads `adp_consensus_pipeline.ipynb` from disk and reports **65 cells: 44 markdown, 21
code**. Every structural rule passes: the first cell is `# Introduction`, the last is
`# Conclusion and Next Steps`, all 21 code cells are preceded by an `### Explain` markdown cell and
followed by an `### Interpretation` markdown cell, no two code cells are adjacent, and no code cell
contains an error output or traceback. The stray-`.py` check confirms no companion analysis or
validation script was left behind — all lasting logic is in this notebook, as
`memory/prefer-ipynb-not-py.md` requires.

The prospective-language scan is the one that guards against the most likely failure of a notebook
like this: it rejects forward-looking phrasing — the kind that describes what a cell is going to
produce rather than what it did — anywhere in the 21 Interpretation cells. (The scanned vocabulary is
assembled from fragments in the code above so that the check's own source line is not a tripwire, and
this cell deliberately paraphrases rather than quotes it: an earlier draft of this very paragraph
quoted the trigger phrases verbatim and the scanner correctly failed the notebook. The scanner was
left strict and the prose was rewritten.) Every interpretation above describes output that was actually produced, and a stale one written
before execution would fail this check rather than quietly shipping.

The error-output check excludes this audit cell itself, for a stated reason: this cell raises on a
structural failure, so on the following pass it would detect its own traceback and fail forever. Its
own prior state is reported separately on the line below it. Every other code cell is scanned without
exception.

**On the execution-state checks.** The line printed above states which file state was read. Because
the notebook is executed twice, this cell reads a *completed* notebook on the second pass and can
therefore decide the execution-count, sequential-numbering, stored-output and error checks about a
real executed artifact rather than about intent. On a first pass those four report `PENDING`, which is
honest rather than convenient — the shipped notebook is the pass-2 artifact, where all checks are
decided.

What this audit does **not** prove: that the analysis is correct, or that the interpretations are
*good*. It proves the notebook is structurally sound, self-contained, fully executed, and free of
prospective language. The substantive verdict rests on Stages 6 through 10, and it is stated next.

# Conclusion and Next Steps

## The verdict

**Three findings, ordered by how much weight they can carry.**

**1. The full-population headline is an undrafted-tail artifact and must never be quoted.**
Over every walk-forward row carrying an ADP, the agreement cell hits **79.9%** at threshold 0 and
**92.3%** at `>10` (pooled 2024–2025, universe A). But **86–92% of those agreement calls are players
outside the draftable top 180**, with a median overall ADP of **634 at `t>5`**. Their median season
total is **25.0 half-PPR points** against **120.3** for a drafted player, and the ten largest calls —
Tutu Atwell at 28.2 points, Tai Felton at 4.0, Allen Lazard at 18.0 — are players nobody drafts. Both
projections correctly ordered the noise floor below the draftable range. That is a true statement
about deep ADP being uninformative; it is not a claim about beating a market.

**2. On the drafted board the pattern survives, beats an empirical null, and rests on very few
calls.** Restricted to `adp_overall_rank <= 180`, pooled 2024–2025, universe A: **66.1%** at t>0
(n=162), **85.0%** (n=40), **93.1%** (n=29), **100%** (n=15). The permutation null sits at
**0.483–0.505** across all 48 cells — so the 50% reference is empirically right — and **p = 0.0001**
everywhere, with **0 of 48 cells** failing to clear their own null 95th percentile. But the per-season
picture is not stable: t>0 runs **83.5% (2021), 75.6%, 69.6%, 74.4%, 56.6% (2025)**, and the 2025
Wilson interval **[0.454, 0.671] contains 0.50**. The most recent complete season is the weakest, and
the trend is downward.

**3. The incremental claim over Sleeper alone is not established.** This is the question that decides
whether the model earns a place beside a projection that already exists free. Among Sleeper's own
calls on the drafted board, adding our model's agreement gives, over the full **2021–2025** panel:

| threshold | lift over Sleeper alone | 95% CI | n (agree / no-agree) |
|---|---|---|---|
| **>5** | **+0.068** | **[−0.028, +0.162]** | 105 / 161 |
| **>7.5** | **+0.073** | **[−0.028, +0.172]** | 70 / 128 |
| **>10** | **+0.035** | **[−0.095, +0.156]** | 37 / 77 |

**Every interval crosses zero.** The shorter 2023–2025 and 2024–2025 panels do show positive lifts
clear of zero at the higher thresholds, but they are nested subsets of the panel that fails, they hold
a third to a half as many calls, and the per-season trend runs against them. Sleeper's *unaided*
drafted-board calls already hit 77.0% / 81.3% / 85.7% at those thresholds — a high bar that our
agreement does not demonstrably raise.

**Stated plainly: the agreement cell picks the right side of ADP well above chance, but on the drafted
board this study cannot show that our model adds anything to Sleeper's projection on its own.** The
lift over *our model alone* is large and stable (+0.21 to +0.48) — the weak direction of the claim,
and unsurprising given the shipped models beat Sleeper at no position.

## What the study rejects

- **"My model beats ADP."** It does not. Agreement is a joint filter requiring Sleeper, and the
  model's own unaided calls hit 43.9–58.6% on the drafted board.
- **"My model beats Sleeper."** Contradicted by the projection build itself at every position
  (RB ρ +0.689, WR +0.736, TE +0.734, QB +0.695, all below Sleeper's).
- **"A 90%+ hit rate."** Only true of a cell that is nine-tenths undrafted players.
- **"It adds signal on top of Sleeper."** Not demonstrated — CIs cross zero on the five-season drafted
  panel at every threshold above 0.
- **"100% at 10+ spots."** Fifteen calls in two seasons.
- **Any per-player 2026 call.** No player-level claim is validated at any threshold.

## Limitations

- **Post-hoc and descriptive.** Not pre-registered, no accept/reject gate, no one-shot discipline.
  The threshold grid was fixed in advance by the request, but the study itself was designed after the
  models shipped.
- **The drafted/undrafted split governs everything.** Any figure taken from `all_adp` is dominated by
  players priced at picks 400–700 whose season totals are near zero.
- **Small cells.** 40 agreement calls at `t>5` across two full seasons on the drafted board — about
  **20 per season** — falling to 15 at `t>10`. Three of five per-season cells at `t>10` are flagged
  `TOO_SMALL`.
- **Not stable across seasons**, and weakest in the most recent one.
- **Freshness is a real, measured component.** The stored Sleeper projection is a week-1-eve snapshot
  while Sleeper ADP is an untimestamped summer aggregate (`PREREGISTRATION.md`, SLEEPER FRESHNESS
  ASYMMETRY). Against the dated Underdog final window the signal attenuates in **all 24 comparisons**
  without exception (five-season drafted board: 72.2% → 68.3% at t>0, 83.8% → 78.3% at t>5) — but does
  not collapse. The share attributable to timing cannot be measured here, and the Underdog format
  difference is uncontrolled.
- **The two systems are not independent.** Sleeper is public and our model trains on overlapping
  information; no "two independent signals" reading is available.
- **Rank universe is settled but population is not.** No conclusion moves between universe A and B;
  every conclusion moves between `all_adp` and `drafted_top180`.
- **Sleeper vintage.** 19 rows carry a Sleeper value in the rebuilt season dataset that was absent
  when the walk-forward files were written. This study uses the build-time column, so those rows sit
  in rank denominators but express no disagreement. Including them would push the headline in the
  flattering direction.
- **Stored `adp_pos_rank` is unusable** (matches a reconstruction on only 48.7% of rows, because it
  was ranked over a larger external universe), so all ranks are rebuilt in-population and are
  population-dependent by construction.

## Video-safe language

**Safe to say:**

- "Backtested over 2021–2025, when my model and Sleeper both ranked a *drafted* player at least 8
  spots away from his ADP in the same direction, he finished on that side about **89%** of the time —
  on **70 calls across five seasons**."
- "That's roughly 20 calls a season at the looser bar and fewer than 10 at the tighter one."
- "It's a late-draft signal: Sleeper's projection is a week-1-eve snapshot and ADP is a summer
  average, so part of it is news the market hadn't finished pricing."
- "Against a market with an actual date on it, the effect is smaller but still there."
- "Backtested, not live-validated. The first real test is the end of 2026."
- "Where the two projections disagree with each other, there's nothing to read."

**Not safe to say** — each is contradicted by a number in this notebook:

- ~~"My model beats ADP"~~ / ~~"beats Sleeper"~~ / ~~"adds signal on top of Sleeper"~~
- ~~"90% hit rate"~~ or ~~"100% at 10+ spots"~~ without the drafted-board restriction *and* the call count
- Any buy / fade / steal / reach / bust call on a named 2026 player, any tier name, or any projected
  hit rate for 2026

## Next steps

1. **The only honest validation is forward.** Freeze the 2026 agreement calls **before** the season —
   the drafted-board cell at `t>5` and `t>7.5`, roughly 20 and 10 players — with the date and the ADP
   snapshot recorded, then grade them after the season. That is a genuine out-of-sample test with no
   post-hoc freedom, and it is the only thing that can promote any of this above "backtested".
2. **Pre-register it if it is to count.** Under `cowork-research-methodology`, a claim this study
   cannot establish needs a written prereg with the threshold, population, and decision rule frozen
   in advance. Reusing this notebook's thresholds without one would be gate-shopping against a panel
   whose results are now known.
3. **Do not tune the threshold on this data.** The 2024–2025 t>10 cell reads 100% on 15 calls;
   selecting `>10` because of that would be fitting to fifteen observations.
4. **If a freshness-clean version is wanted**, the dated Underdog panel is the instrument, but it
   needs its own design — matched format, or an explicit format adjustment — before its numbers can be
   read as anything but directional.

The active source of truth for this study is this notebook. `artifacts/` holds the machine-readable
outputs it generates; `archive/original_2026-08-02/` holds the original run, byte-for-byte.